# 🧠 22 · Neural networks: from logistic regression to deep learning on DNA
### Neurons as logistic regressions, hidden layers for epistasis, backpropagation by hand and in PyTorch, a 1-D CNN that discovers a transcription-factor motif, graph networks, autoencoders and recurrent networks

**BiologyToML** · Level 1 · 02-620 core · Part E — Bayesian learning, neural networks and model selection · Module 22 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/22_neural_networks.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, PyTorch (CPU), Matplotlib, pandas (tables only) |
| **Prerequisites** | Modules 03 (matrices), 04 (derivatives, chain rule, gradient descent), 05 (likelihood, MLE/MAP), 07 (regularization) and 09 (logistic regression, cross-entropy); Module 14 (PCA) for Section 10 |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Neural networks: the biological motivation, logistic regression as the basic neuron unit, two-layer feed-forward networks for classification and regression, linear vs nonlinear decision boundaries, activation functions (sigmoid, tanh, ReLU), deep networks and learned intermediate representations; architectures — convolutional networks (receptive fields, tied weights, pooling; CNNs for transcription-factor binding sites), graph convolutional networks (drug interactions), autoencoders, variational autoencoders, neural networks vs probabilistic graphical models, recurrent networks; learning — loss minimisation, gradient descent vs stochastic and mini-batch gradient descent, the backpropagation algorithm |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED** (labelled as such in every plot): the epistasis data, the DNA sequences with planted motifs,
> the drug–protein graph, and the leukaemia expression data of Section 10, which come from the toolkit's
> `simulate_leukemia_expression` (72 patients × 3,571 genes, shaped like Golub et al. 1999 — the values are not the
> paper's). All networks are tiny on purpose, so that every number can be checked by hand; they teach mechanisms, not
> state-of-the-art genomics models.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why a single artificial neuron *is* a logistic-regression model, and why a hidden layer of neurons can draw nonlinear decision boundaries (XOR, epistasis) that logistic regression cannot.
2. **Compute** a forward pass with explicit matrix shapes, count parameters, and **compare** the sigmoid, tanh and ReLU activations and their derivatives (saturation and vanishing gradients).
3. **Derive** backpropagation for a two-layer network with the chain rule — the classic squared-error rule $\delta_k=o_k(1-o_k)(t_k-o_k)$ and the cross-entropy rule $\delta_k=t_k-o_k$ — and **verify** it with finite differences.
4. **Implement** full-batch gradient descent, stochastic gradient descent and mini-batch SGD, and **diagnose** overfitting and control it with network size, early stopping and weight decay.
5. **Rebuild** the network in PyTorch and **check** that automatic differentiation reproduces your hand-written gradients.
6. **Build** and **train** a 1-D convolutional network on one-hot DNA that finds a planted E-box motif, **interpret** its learned filter, and **explain** why weight tying beats a fully connected network here.
7. **Describe** graph convolutional networks, autoencoders (linear autoencoder = PCA, verified on simulated leukaemia data), variational autoencoders and recurrent networks, and **connect** neural networks to the graphical models of Part D.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · A neuron is logistic regression | weighted sum + sigmoid; gene-regulatory AND/OR logic | 15 min |
| 2 · Why hidden layers | compensatory mutations (XOR); a hand-built 2-2-1 network | 15 min |
| 3 · Forward pass and activations | matrix shapes, parameter counts, sigmoid/tanh/ReLU, saturation | 15 min |
| 4 · Loss and backpropagation | derive, implement, gradient-check, train on epistasis data | 25 min |
| 5 · GD vs SGD vs mini-batch | loss curves per epoch and per second | 10 min |
| 6 · Overfitting and capacity | hidden units, early stopping, weight decay, local minima | 15 min |
| 7 · The same network in PyTorch | autograd = your backpropagation | 10 min |
| 8 · Convolutional networks on DNA | motif scanning, a trained 1-D CNN, its learned filter | 25 min |
| 9 · Graph neural networks | one message-passing layer on a drug–protein graph | 10 min |
| 10 · Autoencoders, VAEs and RNNs | linear autoencoder = PCA on SIMULATED leukaemia data; bottlenecks; unrolling in time | 20 min |
| 11 · Neural networks vs graphical models | strengths, hybrids, further reading | 5 min |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

A transcription factor such as MYC–MAX or CLOCK–BMAL1 binds DNA at a short word, the **E-box** `CACGTG`. Somewhere in
the 3 billion letters of your genome there are hundreds of thousands of such words, but only some are actually bound
in a given cell type. Can a computer learn, from experimental binding data alone, *which sequence features* predict
binding — without being told the motif?

In 2015 two landmark studies answered yes: **DeepBind** (Alipanahi *et al.*) and **DeepSEA** (Zhou & Troyanskaya)
trained **convolutional neural networks** on hundreds of ChIP-seq and related experiments and learned motif-like
filters directly from raw sequence. The architecture is simple to describe: a 1-D convolution scans the
one-hot-encoded DNA (and its reverse complement), a ReLU and a max-pooling step ask "is the pattern anywhere?", and
fully connected layers make the prediction, which is trained against references from functional experiments. Since
then neural networks have spread through biology — protein-structure prediction, single-cell analysis (variational
autoencoders), drug–drug interaction prediction on graphs.

This notebook builds that whole story from the ground up, starting from something you already know. **A single
artificial neuron is exactly the logistic-regression model of [Module 09](09_logistic_regression.ipynb).** Stacking
neurons gives nonlinear boundaries (we use a genetic interaction — epistasis — as the example), training them needs
the chain rule (**backpropagation**), and tying weights across positions gives a **convolutional** network that, in
Section 8, rediscovers the E-box from simulated sequences in a few seconds on a laptop CPU.

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every data set in this notebook is generated
> by code — including the leukaemia expression data of Section 10, which imitate Golub *et al.* (1999). Simulated data
> have a second advantage: we know the truth (the planted motif, the true phenotype rule, the true subtypes), so we can
> check what a network has learned and compare it with the best possible (Bayes-optimal) answer.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)                  # PyTorch has its own random-number generator
torch.set_num_threads(min(2, torch.get_num_threads()))   # small networks: ≤ 2 CPU threads are plenty (and polite)
DEVICE = torch.device("cpu")             # everything here is tiny; a GPU would not make it faster
print("PyTorch", torch.__version__, "| device:", DEVICE, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the logistic function `sigmoid`, `standardize` (z-scores), `accuracy`, `one_hot_dna` (DNA → L × 4 matrix), two plotting helpers and the generator of the SIMULATED leukaemia expression data (shaped like Golub et al. 1999). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import matplotlib.pyplot as plt

LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def one_hot_dna(seq: str) -> np.ndarray:
    """One-hot encode a DNA string as an L × 4 array with columns A, C, G, T (N → all zeros)."""
    lookup = {"A": 0, "C": 1, "G": 2, "T": 3}
    out = np.zeros((len(seq), 4))
    for i, base in enumerate(seq.upper()):
        if base in lookup:
            out[i, lookup[base]] = 1.0
    return out


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'sigmoid', 'standardize', 'accuracy', 'one_hot_dna', 'plot_decision_regions', 'draw_graph']))

---
## 1 · A neuron is logistic regression

> 📍 **Where we are.** In [Module 09](09_logistic_regression.ipynb) you modelled the probability of a binary outcome
> (cancer / healthy) as a sigmoid of a weighted sum of features and fitted the weights by maximum likelihood with
> gradient descent ([Module 04](04_calculus_and_optimization.ipynb)). This section shows that this model is the
> "neuron" from which every neural network is built.

**The biological motivation.** A neuron receives signals on its **dendrites**; each synapse has a strength (and
a sign: excitatory or inhibitory); the **cell body** integrates the inputs; if the integrated signal passes a
threshold at the axon hillock, the neuron **fires** an action potential down its axon. An artificial neuron keeps
only this skeleton: *multiply each input by a weight, add, and pass the sum through a squashing function.*

**The model.** With $n$ input features $X=[X_1,\dots,X_n]$ (elsewhere the course usually writes $J$ for the number
of features; neural-network texts use $n$) and a Boolean output $Y$:

$$P(Y=1\mid X)=\frac{1}{1+\exp\!\big(-w_0-\sum_{i=1}^{n} w_iX_i\big)}=\sigma\Big(w_0+\sum_{i=1}^{n}w_iX_i\Big).$$

*In words:* the neuron computes a **linear combination** $\mathrm{net}=w_0+\sum_i w_iX_i$ of its inputs and
squashes it with the **sigmoid activation** $\sigma$ into a number between 0 and 1, read as the probability
that $Y=1$. Writing $x_0=1$ makes the intercept an ordinary weight: $\mathrm{net}=\sum_{i=0}^{n}w_ix_i$. The parameters of
this neuron are $w_0,w_1,\dots,w_n$ — exactly the logistic-regression weights. The diagram drawn in cell 1.1 is the
standard picture of this computation.

🧮 **Math refresher — the sigmoid.** $\sigma(z)=1/(1+e^{-z})$, where $e\approx2.718$ and $e^{-z}$ is "e to the power
$-z$". Three facts we use all the time: $\sigma(0)=0.5$; $\sigma(z)\to1$ for large positive $z$ and $\to0$ for large
negative $z$; and its derivative is $\sigma'(z)=\sigma(z)\,(1-\sigma(z))$ (derived in Module 04), which is at most
$0.25$ (at $z=0$). The toolkit's `sigmoid` computes it without overflow.

In [ ]:
# ── 1.1 The artificial neuron as a diagram (and a helper that draws whole networks) ──
def draw_net(ax, layers, edge_labels=None, title="", colors=None, layer_names=None, edge_colors=None, r=0.2, dx=1.7,
             fs=8, backward=False):
    """Draw a layered network left → right. `layers` = list of lists of node labels; every node feeds every node of the
    next layer (arrows point right→left if backward=True). edge_labels / edge_colors map (layer, i, j) → text / colour."""
    pos = [[(l * dx, (len(nodes) - 1) / 2 - i) for i in range(len(nodes))] for l, nodes in enumerate(layers)]
    for l in range(len(layers) - 1):
        for i, (x0, y0) in enumerate(pos[l]):
            for j, (x1, y1) in enumerate(pos[l + 1]):
                col = (edge_colors or {}).get((l, i, j), "0.55")
                tip, tail = ((x1 - r, y1), (x0 + r, y0)) if not backward else ((x0 + r, y0), (x1 - r, y1))
                ax.annotate("", xy=tip, xytext=tail, arrowprops=dict(arrowstyle="-|>", color=col, lw=0.9,
                                                                       shrinkA=0, shrinkB=0))
                if edge_labels and (l, i, j) in edge_labels:
                    ax.text(x0 + 0.35 * (x1 - x0), y0 + 0.35 * (y1 - y0), edge_labels[(l, i, j)], fontsize=fs - 1,
                            ha="center", va="center", color=col, bbox=dict(fc="w", ec="none", pad=0.2))
    for l, nodes in enumerate(layers):
        for (x, y), lab in zip(pos[l], nodes):
            ax.add_patch(plt.Circle((x, y), r, fc=(colors or ["#c6dbef"] * len(layers))[l], ec="k", lw=0.8, zorder=3))
            ax.text(x, y, lab, ha="center", va="center", fontsize=fs, zorder=4)
    top = max(len(n) for n in layers) / 2
    for l, name in enumerate(layer_names or []):
        ax.text(l * dx, top + 0.2, name, ha="center", fontsize=fs)
    ax.set_xlim(-0.7, (len(layers) - 1) * dx + 0.7); ax.set_ylim(-top - 0.3, top + 0.9)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=9)
    return pos


fig, ax = plt.subplots(figsize=(9, 3.2))
labels = ["1", "$x_1$", "$x_2$", "⋮", "$x_n$"]
for i, lab in enumerate(labels):                                  # inputs (x0 = 1 carries the intercept)
    ax.add_patch(plt.Circle((0, 2 - i), 0.22, fc="#c6dbef" if lab != "⋮" else "w", ec="k" if lab != "⋮" else "w"))
    ax.text(0, 2 - i, lab, ha="center", va="center", fontsize=8)
    if lab != "⋮":
        ax.annotate("", xy=(2.35, 0), xytext=(0.25, 2 - i), arrowprops=dict(arrowstyle="-|>", color="0.4", lw=0.9))
        w = "$w_n$" if lab == "$x_n$" else f"$w_{i}$"
        ax.text(0.9, (2 - i) * 0.68, w, fontsize=8, color="tab:blue", bbox=dict(fc="w", ec="none", pad=0.2))
ax.text(-0.35, 2, "$x_0$ =", ha="right", va="center", fontsize=8)
ax.add_patch(plt.Circle((2.6, 0), 0.25, fc="#fdd0a2", ec="k")); ax.text(2.6, 0, "Σ", ha="center", va="center", fontsize=12)
ax.text(2.6, -0.5, "linear combination\nnet $=\\sum_{i=0}^{n} w_i x_i$", ha="center", va="top", fontsize=8)
ax.annotate("", xy=(4.3, 0), xytext=(2.85, 0), arrowprops=dict(arrowstyle="-|>", lw=1.2))
ax.add_patch(plt.Rectangle((4.3, -0.45), 1.2, 0.9, fc="#e5f5e0", ec="k"))
zz = np.linspace(-6, 6, 50)
ax.plot(4.35 + 1.1 * (zz + 6) / 12, -0.35 + 0.7 * sigmoid(zz), color="tab:green", lw=2)
ax.text(4.9, -0.6, "sigmoid activation\n$o=\\sigma(\\mathrm{net})=1/(1+e^{-\\mathrm{net}})$", ha="center", va="top", fontsize=8)
ax.annotate("", xy=(6.75, 0), xytext=(5.5, 0), arrowprops=dict(arrowstyle="-|>", lw=1.2))
ax.add_patch(plt.Circle((7.0, 0), 0.25, fc="#fcbba1", ec="k")); ax.text(7.0, 0, "$o$", ha="center", va="center", fontsize=10)
ax.text(7.0, 0.45, "output\n= P(Y = 1 | X)", ha="center", fontsize=8)
ax.set_xlim(-1.0, 8.0); ax.set_ylim(-1.6, 2.5); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("one artificial neuron = logistic regression: parameters $w_0, w_1, \\dots, w_n$", fontsize=10)
plt.tight_layout(); plt.show()

Read the diagram from left to right: each input $x_i$ is multiplied by its weight $w_i$ (the constant input $x_0=1$
carries the intercept $w_0$), the products are summed, and the sum is squashed by the sigmoid. The helper `draw_net`
defined in the same cell will draw whole networks of such neurons in the next sections.

In [ ]:
# ── 1.2 A neuron as a gene-regulatory logic gate ──
def neuron(X, w0, w):
    """Output of one sigmoid neuron: sigma(w0 + X @ w) for each row of X (identical to logistic regression)."""
    return sigmoid(w0 + np.asarray(X, float) @ np.asarray(w, float))


# Two activator transcription factors, each bound (1) or not (0) at a promoter
X_bool = np.array([[0, 0], [1, 0], [0, 1], [1, 1]])
gates = {"AND (needs both activators)": (-15.0, [10.0, 10.0]),
         "OR (either activator suffices)": (-5.0, [10.0, 10.0])}
for name, (w0, w) in gates.items():
    out = neuron(X_bool, w0, w)
    print(f"{name}:  w0 = {w0:+.0f}, w = {w}")
    for x, o in zip(X_bool, out):
        print(f"    TF1 = {x[0]}, TF2 = {x[1]}  →  net = {w0 + x @ w:+5.0f},  P(expressed) = {o:.4f}")

🔍 **What just happened?** With weights $w_1=w_2=10$ and intercept $w_0=-15$, the weighted sum is positive only when
*both* inputs are 1 ($-15+10+10=+5$, probability 0.9933) — an **AND gate**, like a promoter that needs two activators
bound together. Moving the intercept to $-5$ turns it into an **OR gate** (one bound activator gives $+5$). A neuron
is a soft, tunable logic gate; the size of the weights sets how sharp the switch is. Cis-regulatory "logic" of this
kind is a real and well-studied phenomenon (think of the *lac* operon integrating lactose and glucose signals).

Now the crucial question: is there a gene-regulatory function a single neuron **cannot** compute?

In [ ]:
# ── 1.3 Can ONE neuron compute "exactly one input is on" (XOR)? Search 68,921 weight settings ──
t_xor = np.array([0, 1, 1, 0])                                 # XOR truth table for the four rows of X_bool
grid = np.arange(-10, 10.5, 0.5)                               # every w0, w1, w2 in [-10, 10] in steps of 0.5
W0, W1, W2 = np.meshgrid(grid, grid, grid, indexing="ij")
net = W0[..., None] + W1[..., None] * X_bool[:, 0] + W2[..., None] * X_bool[:, 1]   # shape (41, 41, 41, 4)
n_correct = ((net > 0).astype(int) == t_xor).sum(axis=-1)      # rows classified correctly by each weight setting
print(f"weight settings tried: {n_correct.size:,}")
print("best number of XOR rows a single neuron gets right:", n_correct.max(), "of 4")
print("for AND, by comparison:", (((net > 0).astype(int) == np.array([0, 0, 0, 1])).sum(-1)).max(), "of 4")

In [ ]:
# ── 1.4 Why: a neuron's decision boundary is a straight line ──
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
targets = {"AND": np.array([0, 0, 0, 1]), "OR": np.array([0, 1, 1, 1]), "XOR": t_xor}
for ax, (name, t) in zip(axes, targets.items()):
    if name in ("AND", "OR"):
        w0, w = gates[[k for k in gates if k.startswith(name)][0]]
        plot_decision_regions(lambda P: (neuron(P, w0, w) > 0.5).astype(int), X_bool, t, ax=ax, pad=0.6)
        ax.set_title(f"{name}: one neuron separates the classes")
    else:
        for c in (0, 1):
            ax.scatter(*X_bool[t == c].T, s=60, color=["#1f77b4", "#d62728"][c], edgecolor="k", label=f"class {c}")
        ax.set_xlim(-0.6, 1.6); ax.set_ylim(-0.6, 1.6)
        ax.set_title("XOR: no straight line separates them")
    ax.set_xlabel("input 1 (0 = off, 1 = on)"); ax.set_ylabel("input 2")
axes[2].legend(fontsize=7, loc="center")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Among almost 69,000 weight settings, the best single neuron gets **3 of the 4** XOR rows
right, while AND is solved perfectly. The pictures show why. A neuron outputs 0.5 exactly where
$w_0+w_1X_1+w_2X_2=0$, which is a **straight line** in the input plane (a plane or hyperplane with more inputs). That is
why logistic regression always has a **linear decision boundary** (between the regions $Y=1$ and $Y=0$). The XOR classes sit on opposite
corners of a square: no line puts both red corners on one side and both blue corners on the other.

⚠️ **Misconception.** "Logistic regression is nonlinear because the sigmoid is curved." The *probability* is a curved
function of the inputs, but the *boundary* where the decision flips is linear. To get curved boundaries you either
engineer new features by hand (e.g. add $X_1X_2$) or let a network **learn** the features — the subject of Section 2.

✏️ **Your turn.** A promoter is *activated* by TF1 but *repressed* by TF2. Choose $w_0,w_1,w_2$ for a neuron that
outputs ≈1 only for (TF1 = 1, TF2 = 0) and check it with `neuron(X_bool, w0, [w1, w2])`. What sign must $w_2$ have?

---
## 2 · Why hidden layers: XOR and epistasis

> 📍 **Where we are.** One neuron = one straight boundary. Here we connect neurons into the **two-layer feed-forward
> network** and show that the extra ("hidden") layer is what buys nonlinear boundaries.

**A biological XOR: compensatory mutations.** Many RNAs (tRNAs, ribosomal RNAs, riboswitches) fold into helices held
together by base pairs. Suppose a helix contains a G–C pair. Mutating the G to C (mutation A) gives a C–C mismatch and
the RNA misfolds; mutating the C to G (mutation B) gives a G–G mismatch and it also misfolds. But carrying **both**
mutations swaps the pair to C–G, which pairs perfectly again: the double mutant folds normally. The misfolding
phenotype is present **when exactly one of the two mutations is present** — the XOR truth table. Such *compensatory*
(sign) epistasis is how comparative genomics detects conserved RNA structure. (Our version is an idealised,
SIMULATED example.)

**The two-layer feed-forward network.** An **input layer** holds the features, a **hidden layer** of $n_h$ neurons each
computes its own logistic regression of the inputs, and an **output layer** computes logistic regressions of the
hidden outputs. The textbook example (Mitchell 1997) recognises spoken vowels: the inputs are two formant frequencies
F1 and F2 of the sound, and there is one output node for each of ten words of the form h‑vowel‑d (*head*, *hid*, …,
*hood*). In standard notation, for hidden node $j$ and output node $l$:

$$o_{h,j}=\sigma\Big(w_{h,0,j}+\sum_{i=1}^{n}X_i\,w_{h,i,j}\Big),\qquad
  o_{k,l}=\sigma\Big(w_{k,0,l}+\sum_{j=1}^{n_h}o_{h,j}\,w_{k,j,l}\Big).$$

*In words:* hidden neuron $j$ has its own weights $w_{h,i,j}$ (from input $i$ to hidden node $j$) and produces
$o_{h,j}$; output neuron $l$ treats the hidden outputs as *its* features: each neuron has its own set of parameters.
For regression the output node simply omits the sigmoid, $o_{k,l}=w_{k,0,l}+\sum_j o_{h,j}w_{k,j,l}$.

We now set the weights **by hand** to solve XOR: hidden neuron 1 = "at least one mutation" (OR), hidden neuron 2 =
"both mutations" (AND), and the output = "OR but not AND".

In [ ]:
# ── 2.1 A hand-built 2-2-1 network for compensatory mutations ──
W_h_hand = np.array([[10.0, 10.0],      # weights from input 1 (mutation A) to hidden neurons 1 and 2
                     [10.0, 10.0]])     # weights from input 2 (mutation B)
b_h_hand = np.array([-5.0, -15.0])      # hidden 1 behaves like OR, hidden 2 like AND
W_k_hand = np.array([[10.0], [-20.0]])  # output: excited by "at least one", strongly inhibited by "both"
b_k_hand = np.array([-5.0])

O_h = sigmoid(X_bool @ W_h_hand + b_h_hand)        # hidden-layer outputs, one row per genotype
O_k = sigmoid(O_h @ W_k_hand + b_k_hand)           # network output = P(misfolded)
print("mutA mutB | hidden 1 (OR)  hidden 2 (AND) | P(misfolded)   XOR target")
for x, h, o, t in zip(X_bool, O_h, O_k[:, 0], t_xor):
    print(f"  {x[0]}    {x[1]}   |    {h[0]:.3f}          {h[1]:.3f}      |    {o:.3f}          {t}")

fig, ax = plt.subplots(figsize=(6.5, 3))
wl = {(0, i, j): f"{W_h_hand[i, j]:+.0f}" for i in range(2) for j in range(2)} | \
     {(1, j, 0): f"{W_k_hand[j, 0]:+.0f}" for j in range(2)}
pos = draw_net(ax, [["mut A", "mut B"], ["OR", "AND"], ["$o$"]], edge_labels=wl, fs=8, r=0.28, dx=2.2,
               colors=["#c6dbef", "#fdd0a2", "#fcbba1"], layer_names=["input layer", "hidden layer", "output layer"],
               title="the hand-built two-layer network (weights on the arrows, intercepts b next to the units)")
for (x, y), b in zip(pos[1] + pos[2], list(b_h_hand) + list(b_k_hand)):
    ax.text(x, y - 0.42, f"b = {b:+.0f}", ha="center", fontsize=7, color="tab:red")
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 Input space vs hidden space: the hidden layer re-represents the data ──
def hand_net(P):
    """Forward pass of the hand-built network for an (M, 2) array of inputs; returns P(output = 1)."""
    return sigmoid(sigmoid(P @ W_h_hand + b_h_hand) @ W_k_hand + b_k_hand)[:, 0]


fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6))
plot_decision_regions(lambda P: (hand_net(P) > 0.5).astype(int), X_bool, t_xor, ax=a1, pad=0.6,
                      class_names=["folds (0)", "misfolds (1)"])
a1.set_xlabel("mutation A present"); a1.set_ylabel("mutation B present")
a1.set_title("input space: a nonlinear (band-shaped) boundary"); a1.legend(fontsize=7, loc="upper right")
for c in (0, 1):
    a2.scatter(*O_h[t_xor == c].T, s=70, color=["#1f77b4", "#d62728"][c], edgecolor="k")
for lab, h in [("(0,0)", O_h[0]), ("(1,0) and (0,1)", O_h[1]), ("(1,1)", O_h[3])]:
    a2.annotate(lab, h, xytext=(-30, 10), textcoords="offset points", fontsize=8)
hh = np.linspace(-0.1, 1.1, 2)
a2.plot(hh, (5 - 10 * hh) / -20, "k-", lw=0.8)       # output boundary: -5 + 10 h1 - 20 h2 = 0
a2.set_xlim(-0.1, 1.15); a2.set_ylim(-0.1, 1.1)
a2.set_xlabel("hidden 1 = OR detector"); a2.set_ylabel("hidden 2 = AND detector")
a2.set_title("hidden space: now a straight line separates them")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The network outputs ≈0.99 for the single mutants and ≈0.01 for the wild type and the double
mutant — XOR solved. The right panel shows *how*: the hidden layer maps the four genotypes to new coordinates
(OR-detector, AND-detector). In those coordinates (1,0) and (0,1) land on the same point, and a single straight line —
the output neuron — separates the classes. Back in the original input space that line becomes a **band**: a nonlinear
boundary. (In the vowel example, a trained network carves the (F1, F2) plane into ten curved regions, one per word.)

This is the central idea of neural networks: **hidden layers learn intermediate representations.** Here we designed the
representation; in Section 4 the network will discover a similar one by itself from noisy data.

⚠️ **Counting layers.** We call this a *two-layer* network: two layers of weights (input→hidden and
hidden→output); the input layer just holds the data. Some books count three layers. Always count the weight matrices.

✏️ **Your turn.** Change only `W_k_hand` and `b_k_hand` so that the network computes **XNOR** (output 1 when the helix
can fold: both or neither mutation). Hint: flip the signs of the output weights and the intercept.

---
## 3 · The forward pass, parameter counts and activation functions

> 📍 **Where we are.** Section 2 used two hidden neurons and four inputs. Real networks have thousands of neurons and
> samples, so we write the forward pass with matrices ([Module 03](03_linear_algebra_for_data.ipynb)) and look at the
> nonlinear "activation" functions that make networks more than linear models.

**The forward pass in matrix form.** Put $N$ samples in the rows of $X$ ($N\times n$). Collect the hidden weights in
$W_h$ ($n\times n_h$, column $j$ = weights of hidden neuron $j$), the hidden intercepts in $b_h$ (length $n_h$), and
similarly $W_k$ ($n_h\times n_{\text{out}}$) and $b_k$. Then

$$O_h=\sigma\!\left(XW_h+b_h\right)\quad(N\times n_h),\qquad O_k=\sigma\!\left(O_hW_k+b_k\right)\quad(N\times n_{\text{out}}).$$

*In words:* one matrix product computes every hidden neuron's weighted sum for every sample at once; adding $b_h$
adds each neuron's intercept to its column; $\sigma$ is applied to every entry. Row $m$ of $O_k$ holds the network's
outputs for sample $m$.

🧮 **Math refresher — shapes.** An $(a\times b)$ matrix times a $(b\times c)$ matrix gives an $(a\times c)$ matrix;
the inner sizes must match. Adding a vector of length $c$ to an $(a\times c)$ matrix adds it to every row
("broadcasting" in NumPy). Tracking shapes is the single most useful debugging habit in deep learning.

In [ ]:
# ── 3.1 Forward pass with explicit shapes and a parameter count ──
def init_params(n_in, n_hidden, n_out, rng, scale=1.0):
    """Small random weights and zero intercepts (backpropagation starts from 'all weights small random numbers')."""
    return {"W_h": rng.normal(0, scale, (n_in, n_hidden)), "b_h": np.zeros(n_hidden),
            "W_k": rng.normal(0, scale, (n_hidden, n_out)), "b_k": np.zeros(n_out)}


def forward(params, X):
    """Two-layer sigmoid network (Section 2). Returns hidden outputs O_h (N × n_h) and outputs O_k (N × n_out)."""
    O_h = sigmoid(X @ params["W_h"] + params["b_h"])
    O_k = sigmoid(O_h @ params["W_k"] + params["b_k"])
    return O_h, O_k


def n_parameters(params):
    """Total number of adjustable numbers (weights + intercepts)."""
    return sum(v.size for v in params.values())


demo = init_params(n_in=2, n_hidden=3, n_out=1, rng=rng)
X_demo = rng.normal(size=(5, 2))                       # 5 samples, 2 features
O_h_demo, O_k_demo = forward(demo, X_demo)
print("X", X_demo.shape, "@ W_h", demo["W_h"].shape, "→ O_h", O_h_demo.shape, "@ W_k", demo["W_k"].shape,
      "→ O_k", O_k_demo.shape)
print("parameters of this 2-3-1 network:", n_parameters(demo), "= 2·3 + 3 + 3·1 + 1")
big = {"W_h": np.empty((20000, 100)), "b_h": np.empty(100), "W_k": np.empty((100, 5)), "b_k": np.empty(5)}
print(f"a network on 20,000 gene-expression inputs with 100 hidden units and 5 outputs: {n_parameters(big):,} parameters")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[1, 1.9]))
draw_net(axes[0], [["$x_1$", "$x_2$"], ["$h_1$", "$h_2$", "$h_3$"], ["$o$"]], colors=["#c6dbef", "#fdd0a2", "#fcbba1"],
         layer_names=["$X$: N × 2", "$O_h$: N × 3", "$O_k$: N × 1"], title="2-3-1 network: $W_h$ is 2 × 3, $W_k$ is 3 × 1")
draw_net(axes[1], [[""] * 4, [""] * 6, [""] * 6, [""] * 6, [""] * 3], r=0.16, dx=1.9,
         colors=["#c6dbef", "#fdd0a2", "#fdd0a2", "#fdd0a2", "#fcbba1"],
         layer_names=["input", "hidden 1", "hidden 2", "hidden 3", "output"],
         title="a deep feed-forward network: many hidden layers, many nodes per layer")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The shapes chain exactly as the refresher predicts: $(5\times2)(2\times3)\to(5\times3)$, then
$(5\times3)(3\times1)\to(5\times1)$. A toy 2-3-1 network has 13 parameters, but one hidden layer of 100 units on
20,000 genes already has about **two million** — neural networks potentially have millions of parameters to estimate. Keep
this number in mind when you read a paper that trains such a network on 200 patients: it is the overfitting problem
of [Module 07](07_ridge_lasso_sparse_regression.ipynb) on steroids (Section 6 and [Module 23](23_model_selection.ipynb)).

**Alternative activation functions** (also called transfer functions). The sigmoid is not the only choice:

$$\sigma(z)=\frac{1}{1+e^{-z}},\qquad \tanh(z)=\frac{e^{z}-e^{-z}}{e^{z}+e^{-z}}=2\sigma(2z)-1,\qquad \mathrm{ReLU}(z)=\max(0,z).$$

*In words:* the sigmoid squashes into (0, 1), tanh into (−1, 1) (it is a stretched, centred sigmoid), and the
**rectified linear unit** passes positive inputs unchanged and zeroes negative ones. Their derivatives are
$\sigma'=\sigma(1-\sigma)\le0.25$, $\tanh'=1-\tanh^2\le1$, and $\mathrm{ReLU}'=1$ for $z>0$ and $0$ for $z<0$.

⚠️ **Sign conventions.** Some texts write the logistic model as $1/(1+\exp(w_0+\sum_iw_iX_i))$, without the minus signs.
That is the same family of models with all weights negated; this notebook always uses $\sigma(z)=1/(1+e^{-z})$ with
$z=w_0+\sum_iw_iX_i$, so that positive weights push towards $Y=1$.

In [ ]:
# ── 3.2 Sigmoid, tanh and ReLU and their derivatives ──
z = np.linspace(-6, 6, 400)
acts = {"sigmoid": (sigmoid(z), sigmoid(z) * (1 - sigmoid(z))),
        "tanh": (np.tanh(z), 1 - np.tanh(z) ** 2),
        "ReLU": (np.maximum(0, z), (z > 0).astype(float))}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.3))
for (name, (f, df)), col in zip(acts.items(), ["tab:blue", "tab:orange", "tab:green"]):
    a1.plot(z, f, color=col, label=name)
    a2.plot(z, df, color=col, label=name)
a1.set_ylim(-1.2, 3); a1.set_title("activation functions")
a2.set_title("their derivatives: flat regions = saturation")
for a in (a1, a2):
    a.axhline(0, color="k", lw=0.5); a.set_xlabel("weighted input z to the unit"); a.legend()
a1.set_ylabel("output"); a2.set_ylabel("slope d(output)/dz")
plt.tight_layout(); plt.show()
print("largest slope: sigmoid %.3f | tanh %.3f | ReLU %.0f" % tuple(df.max() for f, df in acts.values()))
print("check tanh(z) = 2·sigmoid(2z) − 1:", np.allclose(np.tanh(z), 2 * sigmoid(2 * z) - 1))

In [ ]:
# ── 3.3 Why saturation matters: gradients through a deep stack of units ──
rng = np.random.default_rng(SEED + 3)
depths = [1, 2, 5, 10, 20]
z_deep = rng.normal(0, 2, (5000, max(depths)))     # typical weighted inputs met by a signal in 5,000 random networks
slopes = {"sigmoid": sigmoid(z_deep) * (1 - sigmoid(z_deep)), "tanh": 1 - np.tanh(z_deep) ** 2,
          "ReLU": (z_deep > 0).astype(float)}
print("median size of the chain-rule product of slopes after L layers (all weights = 1):")
print("   L  " + "".join(f"{name:>12s}" for name in slopes))
for L in depths:
    print(f"  {L:2d}  " + "".join(f"{np.median(np.prod(s[:, :L], axis=1)):12.2e}" for s in slopes.values()))
print("fraction of ReLU paths that stay fully open after 10 layers:", np.mean(np.prod(slopes["ReLU"][:, :10], 1) > 0))

🔍 **What just happened?** The sigmoid's slope is never above 0.25 and is almost zero when $|z|$ is large — the unit
is **saturated** (it saturates to 1 or 0): changing its weights barely changes its output. Backpropagation (next section)
multiplies one such slope per layer, so through 10 sigmoid layers the typical product is around $10^{-9}$: the
early layers receive essentially no learning signal. This is the **vanishing-gradient problem**. tanh is better
(slope up to 1) but still saturates. ReLU has slope exactly 1 wherever it is active, so the gradient passes unchanged
along open paths — no saturation, so the output is more sensitive to weight changes — which is why ReLU became the default in
deep networks. Its price: each unit is closed about half of the time, so the median product is 0 and only about
$0.5^{10}\approx0.1\%$ of random 10-layer paths stay fully open (that is why real networks have many units per layer, so
that *some* paths are always open), and a unit whose input is always negative never learns again (a "dead ReLU").

✏️ **Your turn.** Add a **leaky ReLU**, $\max(0.01z, z)$, to `slopes` in cell 3.3. What happens to the median product
and to the fraction of open paths? (Leaky ReLU is one common fix for dead units.)

### 📘 Lecture notes — from a neuron to a deep network

**Neuron = logistic regression.** For inputs $X=[X_1,\dots,X_n]$ and Boolean $Y$:
$P(Y=1\mid X)=1/\big(1+\exp(-w_0-\sum_iw_iX_i)\big)=\sigma(w_0+\sum_iw_iX_i)$ — a linear combination
$\mathrm{net}=\sum_{i=0}^nw_ix_i$ ($x_0=1$) followed by the sigmoid activation; parameters $w_0,\dots,w_n$. Inspired by
biological neurons that integrate weighted synaptic inputs and fire above a threshold.

**Two-layer feed-forward network.** Input layer → hidden layer → output layer; each node has its own parameters:
$$o_{h,j}=\sigma\Big(w_{h,0,j}+\sum_{i=1}^{n}X_iw_{h,i,j}\Big),\qquad o_{k,l}=\sigma\Big(w_{k,0,l}+\sum_{j=1}^{n_h}o_{h,j}w_{k,j,l}\Big)$$
for classification, or $o_{k,l}=w_{k,0,l}+\sum_jo_{h,j}w_{k,j,l}$ (no sigmoid) for regression. Example: vowel
recognition from two formant frequencies (F1, F2) with one output per word (*head, hid, …, hood*).

**Boundaries.** Logistic regression separates $Y=1$ from $Y=0$ with a straight line (hyperplane); a network with a
hidden layer draws nonlinear boundaries (XOR, epistasis; curved regions in the vowel plane).

**Activation functions.** Sigmoid (0 to 1), $\tanh$ (−1 to +1), ReLU $\max(0,z)$: no saturation to 1/0 (sigmoid)
or ±1 (tanh) for positive inputs, so the output is more sensitive to weight changes and gradients do not vanish.

**Deep networks.** Highly nonlinear regression/classification, vector-valued inputs and outputs, many hidden layers
and many nodes per layer, potentially millions of parameters; **hidden layers learn intermediate representations**.
Parameter count of an $n$–$n_h$–$n_{out}$ network: $(n+1)n_h+(n_h+1)n_{out}$.

> 🎓 **Exam tip.** Be able to (i) write a sigmoid neuron as logistic regression, (ii) count the parameters of an $n$–$n_h$–$n_{out}$
network (don't forget the intercepts), (iii) explain why a hidden layer is needed for XOR, and (iv) state why ReLU
avoids saturation. A common mistake is to say logistic regression has a curved boundary.

---
## 4 · Loss functions and backpropagation

> 📍 **Where we are.** We can compute a network's output for given weights. Learning means choosing the weights to
> **minimise a loss function** by gradient descent (Module 04). The only hard part is the gradient — and the
> chain rule organised cleverly, **backpropagation**, makes it cheap: iterate between a forward propagation (compute
> the outputs from the inputs and the current weights) and a backward propagation (propagate the errors backward and
> compute the gradient).

**Two losses.** For one sample with targets $t_k\in\{0,1\}$ and outputs $o_k$:

$$E_{\text{sq}}=\tfrac12\sum_{k}(t_k-o_k)^2,\qquad
  L_{\text{CE}}=-\sum_k\big[t_k\log o_k+(1-t_k)\log(1-o_k)\big].$$

*In words:* the **squared error** (used in the classic statement of backpropagation, Mitchell 1997) penalises the distance between
target and output; the **cross-entropy** is the negative log-likelihood of a Bernoulli model in which $o_k$ is the
probability that $t_k=1$ — the same loss as logistic regression in Module 09. Averaging over the $N$ training samples
gives the training loss.

🔗 **The course's two-step template.** A classification network is a *probability model* $P(Y=1\mid X;w)=o_k$ (step 1);
minimising the cross-entropy is **maximum likelihood** (step 2). Adding a penalty $\lambda\|w\|^2$ ("weight decay",
Section 6) is **MAP estimation** with a Gaussian prior on the weights ([Module 05](05_likelihood_mle_map.ipynb),
[Module 07](07_ridge_lasso_sparse_regression.ipynb)). Networks are discriminative models, like logistic regression.

🧮 **Math refresher — the chain rule.** If $E$ depends on $o$, $o$ on $\mathrm{net}$, and $\mathrm{net}$ on $w$, then
$$\frac{\partial E}{\partial w}=\frac{\partial E}{\partial o}\cdot\frac{\partial o}{\partial\mathrm{net}}\cdot\frac{\partial\mathrm{net}}{\partial w}.$$
*In words:* the sensitivity of the loss to a weight is the product of the sensitivities along the path from the weight
to the loss. The curly $\partial$ ("partial derivative") means: change this one variable and hold all others fixed.

### 4.1 Deriving the backpropagation rules (squared error)

Take one sample. Output unit $k$ has $\mathrm{net}_k=\sum_h w_{hk}o_h$ (the sum includes the intercept via $o_0=1$) and
$o_k=\sigma(\mathrm{net}_k)$. Define the **error term** of a unit as minus the derivative of the loss with respect to its
weighted input, $\delta_j=-\partial E/\partial\mathrm{net}_j$.

**Output units.** By the chain rule, with $\partial E/\partial o_k=-(t_k-o_k)$ and $\partial o_k/\partial\mathrm{net}_k=o_k(1-o_k)$:

$$\delta_k=-\frac{\partial E}{\partial o_k}\frac{\partial o_k}{\partial\mathrm{net}_k}=o_k(1-o_k)(t_k-o_k).$$

*In words:* the output unit's error is the miss $(t_k-o_k)$ scaled by how responsive the sigmoid currently is — the
first backward step, which propagates the error from the targets $t_k$ to the parameters of the output neurons.

**Hidden units.** Hidden unit $h$ influences the loss only through the output units it feeds, so we sum over them:

$$\delta_h=-\frac{\partial E}{\partial\mathrm{net}_h}=-\sum_{k}\frac{\partial E}{\partial\mathrm{net}_k}\frac{\partial\mathrm{net}_k}{\partial o_h}\frac{\partial o_h}{\partial\mathrm{net}_h}
=o_h(1-o_h)\sum_{k\in\text{outputs}}w_{hk}\,\delta_k.$$

*In words:* a hidden unit's error is the output errors **sent backwards** through the same weights that carried the
signal forwards, scaled by the hidden unit's own sigmoid slope — the second backward step, from the output neurons to
the parameters of the hidden neurons. This backward reuse of the forward weights is
why the algorithm is called back-*propagation*.

**Weight updates.** Since $\partial\mathrm{net}_j/\partial w_{ij}=x_i$ (the input arriving at unit $j$ from unit $i$),
$\partial E/\partial w_{ij}=-\delta_jx_i$, and a gradient-descent step with step size $\eta$ is

$$w_{ij}\leftarrow w_{ij}+\eta\,\Delta w_{ij},\qquad \Delta w_{ij}=\delta_j\,x_i.$$

*In words:* move each weight in proportion to the error at its destination times the activity at its source (a
gradient-descent update with step size $\eta$; $w_{ij}$ is the weight from unit $i$ to unit $j$).

**The algorithm.** Initialise all weights to small random numbers. Iterate until convergence: for each training
sample, (1) forward propagation — compute the network outputs $o_k$ for the input $x$; (2) for each output unit,
$\delta_k\leftarrow o_k(1-o_k)(t_k-o_k)$; (3) for each hidden unit, $\delta_h\leftarrow o_h(1-o_h)\sum_k w_{hk}\delta_k$;
(4) update every weight, $w_{ij}\leftarrow w_{ij}+\eta\,\delta_jx_i$. Cell 4.2 runs exactly these steps and draws them.
The **plus** sign is correct because $\delta$ was defined as *minus* the gradient.

**Cross-entropy instead.** With $L_{\text{CE}}$, $\partial L/\partial o_k=-(t_k-o_k)/(o_k(1-o_k))$, and the sigmoid slope
cancels: $\delta_k=t_k-o_k$. The hidden-unit rule is unchanged. A confidently wrong output ($o_k\approx0$, $t_k=1$) now
gets a large error signal instead of a tiny one — the practical reason cross-entropy trains classifiers faster.

In [ ]:
# ── 4.2 The backpropagation algorithm, literally: one stochastic update for one sample ──
def sq_error(params, X, T):
    """Mean over samples of the squared error 1/2 * sum_k (t_k - o_k)^2."""
    return float(np.mean(0.5 * np.sum((T - forward(params, X)[1]) ** 2, axis=1)))


def backprop_step(params, x, t, eta):
    """Update the weights IN PLACE for one sample x (length n) with targets t (length n_out): the classic algorithm."""
    o_h, o_k = (a[0] for a in forward(params, x[None, :]))       # forward propagation
    delta_k = o_k * (1 - o_k) * (t - o_k)                         # for each output unit k
    delta_h = o_h * (1 - o_h) * (params["W_k"] @ delta_k)         # for each hidden unit h: sum_k w_hk delta_k
    params["W_k"] += eta * np.outer(o_h, delta_k)                 # Δw_hk = δ_k · o_h  (input to unit k from unit h)
    params["b_k"] += eta * delta_k                                # intercepts: weights from a constant input 1
    params["W_h"] += eta * np.outer(x, delta_h)                   # Δw_ih = δ_h · x_i
    params["b_h"] += eta * delta_h
    return delta_k, delta_h


rng = np.random.default_rng(SEED + 4)
p = init_params(2, 3, 1, rng)
x1, t1 = np.array([1.0, 0.0]), np.array([1.0])                   # a single mutant: target 'misfolds' = 1
before = sq_error(p, x1[None], t1[None])
o_h1, o_k1 = (a[0] for a in forward(p, x1[None]))               # forward values before the update (for the picture)
dk, dh = backprop_step(p, x1, t1, eta=0.5)
print("δ_k (output) =", np.round(dk, 4), "| δ_h (hidden) =", np.round(dh, 4))
print(f"squared error on this sample: before {before:.5f} → after one update {sq_error(p, x1[None], t1[None]):.5f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
draw_net(axes[0], [[f"{v:.0f}" for v in x1], [f"{v:.2f}" for v in o_h1], [f"{o_k1[0]:.2f}"]], r=0.28, dx=2.0,
         edge_colors={(l, i, j): "tab:blue" for l in range(2) for i in range(3) for j in range(3)},
         colors=["#c6dbef", "#fdd0a2", "#fcbba1"], layer_names=["inputs $x_i$", "hidden $o_h$", f"output $o_k$ (t = {t1[0]:.0f})"],
         title="1. forward propagation: compute the outputs from the inputs")
draw_net(axes[1], [["", ""], [f"{v:+.3f}" for v in dh], [f"{dk[0]:+.3f}"]], r=0.28, dx=2.0, backward=True, fs=7,
         edge_colors={(l, i, j): "tab:red" for l in range(2) for i in range(3) for j in range(3)},
         colors=["#ffffff", "#fdd0a2", "#fcbba1"], layer_names=["", "$\\delta_h$", "$\\delta_k = o_k(1-o_k)(t_k-o_k)$"],
         title="2. backward propagation: send the errors back, then $\\Delta w_{ij}=\\delta_j x_i$")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.3 Vectorised gradients for a whole batch (both losses) ──
def ce_loss(params, X, T):
    """Mean cross-entropy (negative Bernoulli log-likelihood per sample)."""
    O = np.clip(forward(params, X)[1], 1e-12, 1 - 1e-12)
    return float(-np.mean(np.sum(T * np.log(O) + (1 - T) * np.log(1 - O), axis=1)))


def grads(params, X, T, loss="ce"):
    """Gradient of the mean loss w.r.t. every parameter, by backpropagation (rows of X = samples)."""
    O_h, O_k = forward(params, X)
    G_k = (O_k - T) if loss == "ce" else -(T - O_k) * O_k * (1 - O_k)   # = -δ_k for every sample (rows)
    G_k = G_k / len(X)                                                   # we minimise the MEAN loss
    G_h = (G_k @ params["W_k"].T) * O_h * (1 - O_h)                      # = -δ_h: errors sent backwards
    return {"W_k": O_h.T @ G_k, "b_k": G_k.sum(0), "W_h": X.T @ G_h, "b_h": G_h.sum(0)}


def numerical_grads(params, X, T, loss_fn, eps=1e-6):
    """Central finite differences (L(w+eps) - L(w-eps)) / (2 eps), one parameter at a time."""
    out = {}
    for name, value in params.items():
        g = np.zeros_like(value)
        for idx in np.ndindex(value.shape):
            old = value[idx]
            value[idx] = old + eps; up = loss_fn(params, X, T)
            value[idx] = old - eps; down = loss_fn(params, X, T)
            value[idx] = old
            g[idx] = (up - down) / (2 * eps)
        out[name] = g
    return out

In [ ]:
# ── 4.4 Gradient check: backpropagation vs finite differences ──
rng = np.random.default_rng(SEED + 5)
p = init_params(2, 4, 1, rng)
Xc, Tc = rng.normal(size=(30, 2)), rng.integers(0, 2, (30, 1)).astype(float)
for loss, fn in [("ce", ce_loss), ("sq", sq_error)]:
    bp, fd = grads(p, Xc, Tc, loss), numerical_grads(p, Xc, Tc, fn)
    rel = max(np.max(np.abs(bp[k] - fd[k]) / (np.abs(bp[k]) + np.abs(fd[k]) + 1e-12)) for k in p)
    print(f"{'cross-entropy' if loss == 'ce' else 'squared error':14s}: largest relative difference = {rel:.1e}")

🔍 **What just happened?** In 4.2 a single update on one sample nudged the weights so that this sample's error
dropped — the recipe works; the two diagrams show the forward values (left) and the error terms sent backwards
(right). In 4.4 the vectorised backpropagation gradients agree with brute-force finite
differences to a relative difference of about $10^{-8}$ or smaller, for both losses. Finite differences need two loss evaluations
*per parameter* (impossible for a million-parameter network); backpropagation gets *all* derivatives from one
forward and one backward pass. That efficiency is why it made deep learning possible (Rumelhart, Hinton & Williams 1986).

🧮 **Math refresher — finite differences.** The derivative is the limit of a slope: $f'(w)\approx\frac{f(w+\varepsilon)-
f(w-\varepsilon)}{2\varepsilon}$ for a small $\varepsilon$. Comparing an analytic gradient with this approximation is the
standard way to catch bugs in hand-written gradients.

### 4.5 Training on noisy epistasis data

Real genotypes are not handed to us as perfect 0/1 values: a genotyping array measures **fluorescence intensities**,
and carriers and non-carriers form noisy clusters. We simulate 400 individuals whose normalised allele signals at the
two loci scatter around 0 (non-carrier) or 1 (carrier) with standard deviation 0.35, and whose phenotype follows the
compensatory-mutation (XOR) rule of Section 2. Because we know how the data were made, we can also compute the
**Bayes-optimal** accuracy — the best any classifier could do ([Module 23](23_model_selection.ipynb), Section 9).

In [ ]:
# ── 4.5 SIMULATED genotyping signals with an XOR phenotype ──
NOISE = 0.35


def simulate_epistasis(n, rng, noise=NOISE):
    """SIMULATED allele signals at two loci (carrier ≈ 1, non-carrier ≈ 0) and the XOR (compensatory) phenotype."""
    a, b = rng.integers(0, 2, n), rng.integers(0, 2, n)            # true carrier status at loci A and B
    X = np.c_[a, b] + rng.normal(0, noise, (n, 2))                 # noisy normalised intensities
    return X, (a ^ b).astype(int)                                  # phenotype iff exactly one mutation


def bayes_posterior(X, noise=NOISE):
    """Exact P(phenotype = 1 | signals) under the simulation model (four equally likely Gaussian clusters)."""
    def dens(c):
        return np.exp(-np.sum((X - np.array(c)) ** 2, axis=1) / (2 * noise ** 2))
    return (dens((1, 0)) + dens((0, 1))) / (dens((0, 0)) + dens((1, 0)) + dens((0, 1)) + dens((1, 1)))


rng = np.random.default_rng(SEED + 6)
X_tr, y_tr = simulate_epistasis(400, rng)
X_te, y_te = simulate_epistasis(5000, rng)
T_tr = y_tr[:, None].astype(float)
print("training set:", X_tr.shape, "| phenotype frequency:", y_tr.mean().round(3))
print("Bayes-optimal test accuracy (knows the true model):", round(accuracy(y_te, bayes_posterior(X_te) > 0.5), 3))

In [ ]:
# ── 4.6 Logistic regression vs a two-layer network, both trained by full-batch gradient descent ──
def train(params, X, T, lr, epochs, batch_size=None, rng=None, weight_decay=0.0, X_val=None, T_val=None, loss="ce"):
    """Minimise the mean loss by (mini-batch) gradient descent. Returns per-epoch training (and validation) losses."""
    loss_fn = ce_loss if loss == "ce" else sq_error
    history = {"train": [], "val": [], "seconds": []}
    t0, N = time.time(), len(X)
    for _ in range(epochs):
        order = rng.permutation(N) if batch_size else np.arange(N)
        for start in range(0, N, batch_size or N):
            idx = order[start:start + (batch_size or N)]
            g = grads(params, X[idx], T[idx], loss)
            for name in params:
                decay = weight_decay * params[name] if name.startswith("W") else 0.0   # do not shrink intercepts
                params[name] -= lr * (g[name] + decay)
        history["train"].append(loss_fn(params, X, T))
        history["seconds"].append(time.time() - t0)
        if X_val is not None:
            history["val"].append(loss_fn(params, X_val, T_val))
    return history


rng = np.random.default_rng(SEED + 7)
w_lr, b_lr = np.zeros(2), 0.0                                      # logistic regression (Module 09)
for _ in range(3000):
    err = sigmoid(X_tr @ w_lr + b_lr) - y_tr
    w_lr -= 1.0 * X_tr.T @ err / len(y_tr); b_lr -= 1.0 * err.mean()
net6 = init_params(2, 6, 1, rng)                                  # 2 inputs, 6 hidden units, 1 output
hist6 = train(net6, X_tr, T_tr, lr=2.0, epochs=3000)
acc_lr = accuracy(y_te, sigmoid(X_te @ w_lr + b_lr) > 0.5)
acc_nn = accuracy(y_te, forward(net6, X_te)[1][:, 0] > 0.5)
print(f"logistic regression : test accuracy {acc_lr:.3f}")
print(f"2-6-1 network       : test accuracy {acc_nn:.3f}  (final training cross-entropy {hist6['train'][-1]:.3f})")

In [ ]:
# ── 4.7 Decision regions and the learning curve ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
show = slice(0, 400)
plot_decision_regions(lambda P: (sigmoid(P @ w_lr + b_lr) > 0.5).astype(int), X_tr[show], y_tr[show], ax=axes[0])
axes[0].set_title(f"logistic regression: linear boundary\ntest accuracy {acc_lr:.2f} (SIMULATED)")
plot_decision_regions(lambda P: (forward(net6, P)[1][:, 0] > 0.5).astype(int), X_tr[show], y_tr[show], ax=axes[1])
axes[1].set_title(f"2-6-1 network: nonlinear boundary\ntest accuracy {acc_nn:.2f} (SIMULATED)")
for a in axes[:2]:
    a.set_xlabel("allele signal, locus A"); a.set_ylabel("allele signal, locus B")
axes[2].plot(hist6["train"], color="tab:blue")
axes[2].axhline(np.log(2), color="0.5", ls="--", lw=0.8, label="log 2 = guessing")
axes[2].set_xscale("log"); axes[2].set_xlabel("epoch (log scale)"); axes[2].set_ylabel("training cross-entropy")
axes[2].set_title("plateau, then the hidden units 'discover' XOR"); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.8 What did the six hidden units learn? ──
g1, g2 = np.meshgrid(np.linspace(-1, 2, 120), np.linspace(-1, 2, 120))
H_grid = forward(net6, np.c_[g1.ravel(), g2.ravel()])[0]
fig, axes = plt.subplots(2, 3, figsize=(9.5, 5.4))
for j, ax in enumerate(axes.ravel()):
    im = ax.contourf(g1, g2, H_grid[:, j].reshape(g1.shape), levels=np.linspace(0, 1, 11), cmap="viridis")
    ax.scatter([0, 1, 0, 1], [0, 0, 1, 1], c=["w", "r", "r", "w"], edgecolor="k", s=40)
    ax.set_title(f"unit {j + 1} (output weight {net6['W_k'][j, 0]:+.1f})", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
fig.colorbar(im, ax=axes, shrink=0.8, label="unit output $o_h$")
fig.suptitle("each hidden unit is a soft linear 'detector'; red dots = single mutants (SIMULATED)", fontsize=10)
plt.show()

🔍 **What just happened?** Logistic regression is stuck near chance (0.52 test accuracy): with a straight line it
cannot express "exactly one mutation". The 2-6-1 network reaches **0.845**, within one percentage point of the
Bayes-optimal accuracy of 0.853 printed in 4.5 (the clusters overlap, so even a perfect classifier makes mistakes). Its
boundary bends around the two single-mutant clusters. The learning curve shows a typical shape: a long **plateau** near $\log2\approx0.69$ (the network
outputs ≈0.5 for everyone) until the hidden units break symmetry, then a rapid drop.

The six hidden-unit maps show the learned **intermediate representation**: each unit is a sigmoid of a linear function,
i.e. a soft half-plane detector, and the output unit combines them with positive and negative weights — the same kind
of solution we designed by hand in Section 2 (units that switch on for "signal A high", "signal B high", "both high" …), found automatically by gradient
descent. The exact detectors differ from run to run; only their combination matters.

✏️ **Your turn.** Retrain the network with the classic **squared-error** loss: `train(init_params(2, 6, 1,
np.random.default_rng(SEED + 7)), X_tr, T_tr, lr=2.0, epochs=3000, loss="sq")`. Does it reach the same accuracy? Is the
plateau longer or shorter? Relate your answer to the $o_k(1-o_k)$ factor in the squared-error $\delta_k$.

---
## 5 · Gradient descent, stochastic gradient descent and mini-batches

> 📍 **Where we are.** Section 4 used **full-batch gradient descent**: every update averaged the gradient over all 400
> samples. Two alternatives make training on millions of samples possible.

* **Gradient descent:** gradients computed for *all* training samples; one update per pass through the data (epoch).
* **Stochastic gradient descent (SGD):** gradient of *one* sample; $N$ updates per epoch (the backpropagation algorithm
  of Section 4.1 is written this way: "for each sample …").
* **Mini-batch SGD:** gradient summed (or averaged) over a random *subset* of $B$ samples; $\lceil N/B\rceil$ updates per epoch.

With a loss $L(w)=\frac1N\sum_{m=1}^N\ell_m(w)$ and a random mini-batch $\mathcal B$,

$$w\leftarrow w-\eta\,\frac{1}{|\mathcal B|}\sum_{m\in\mathcal B}\nabla\ell_m(w).$$

*In words:* the mini-batch gradient is a noisy but **unbiased** estimate of the full gradient (on average it points the
same way), and it costs only $|\mathcal B|$ sample-gradients instead of $N$. The symbol $\nabla$ ("nabla") denotes the
vector of all partial derivatives. We compare the three on 1,000 SIMULATED individuals.

In [ ]:
# ── 5.1 Same network, same data: GD vs SGD vs mini-batch SGD ──
rng = np.random.default_rng(SEED + 8)
X_big, y_big = simulate_epistasis(1000, rng)
T_big = y_big[:, None].astype(float)
runs = {}
for name, batch, lr, epochs in [("full-batch GD", None, 2.0, 300), ("SGD (batch 1)", 1, 0.1, 10),
                                ("mini-batch (32)", 32, 0.5, 60)]:
    r = np.random.default_rng(SEED + 80)                       # identical starting weights for all three
    p = init_params(2, 4, 1, r)
    runs[name] = train(p, X_big, T_big, lr=lr, epochs=epochs, batch_size=batch, rng=r)
    upd = epochs * (1 if batch is None else int(np.ceil(1000 / batch)))
    print(f"{name:16s}: {epochs:3d} epochs, {upd:6d} updates, final loss {runs[name]['train'][-1]:.3f}, "
          f"{runs[name]['seconds'][-1]:.2f} s")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.3))
for (name, h), col in zip(runs.items(), ["tab:blue", "tab:orange", "tab:green"]):
    a1.plot(np.arange(1, len(h["train"]) + 1), h["train"], color=col, label=name)
    a2.plot(h["seconds"], h["train"], color=col, label=name)
a1.set_xscale("log"); a1.set_xlabel("epochs (passes through the data, log scale)")
a2.set_xlabel("wall-clock time (s)")
for a in (a1, a2):
    a.set_ylabel("training cross-entropy"); a.axhline(np.log(2), color="0.6", ls="--", lw=0.8); a.legend()
a1.set_title("per epoch, SGD makes the most progress"); a2.set_title("per second, vectorised updates win (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Measured **per epoch** (one pass through the data), SGD is by far the fastest learner: it
makes 1,000 small updates per pass and is close to its final loss after 6–8 epochs; mini-batch SGD (32 updates
per pass) needs a few tens of epochs; full-batch GD (one update per pass) sits on the plateau for about 100 epochs and
needs all 300 to get down to a similar loss. Measured **per second**, the picture changes: pure SGD pays for 10,000
tiny Python-level updates and is the slowest, whereas full-batch and mini-batch updates are single efficient matrix
operations and finish in a few hundredths of a second. On this toy problem (1,000 samples that easily fit in memory)
the two vectorised methods are comparable; on a GPU, where a batch of 32 costs about the same time as a batch of 1,
and on data sets too large for full-batch gradients, mini-batches are the clear winner. Notice also that the
mini-batch curve is bumpy (it even goes *up* for a moment early on): each update follows a noisy estimate of the
gradient. (Exact timings depend on your computer; the qualitative picture does not.)

There are two practical reasons why SGD dominates deep learning: the training set may be **too large to fit in
memory** at once, and networks with millions of parameters **need many training samples**, so a full gradient is
expensive. The noise of SGD can even help escape plateaus and poor local minima.

⚠️ **Learning rates are not transferable.** We used η = 2.0, 0.5 and 0.1 for the three methods: smaller batches give
noisier gradients and need smaller steps. Changing the batch size without retuning η is a classic source of "my
network does not train".

✏️ **Your turn.** Try mini-batch sizes 8 and 128 (adjust `lr` if needed). Which reaches loss 0.4 in the fewest seconds?

### 📘 Lecture notes — learning a neural network: loss, SGD and backpropagation

**Learning = minimising a loss function** over all weights, e.g. $E=\frac12\sum_k(t_k-o_k)^2$ or the cross-entropy
(negative log-likelihood).

**Three ways to use the gradient.** *Gradient descent:* gradients for all training samples, weights updated with the
sum (or mean) over all samples — 1 update per epoch. *Stochastic gradient descent:* gradient of one sample per update —
$N$ updates per epoch. *Mini-batch SGD:* gradient summed over a subset of $B$ samples — $\lceil N/B\rceil$ updates per
epoch. SGD is useful when the training set is too large to load into memory and compute the full gradient, and deep
models, with many parameters, need many training samples.

**Backpropagation** (squared error, sigmoid units). Initialise all weights to small random numbers; iterate until
convergence; for each sample:
1. *forward propagation:* compute the outputs $o_k$ (and hidden outputs $o_h$) for input $x$;
2. *output units:* $\delta_k\leftarrow o_k(1-o_k)(t_k-o_k)$ — error from the targets to the output neurons' parameters;
3. *hidden units:* $\delta_h\leftarrow o_h(1-o_h)\sum_{k\in\text{outputs}}w_{hk}\delta_k$ — errors sent backwards;
4. *update:* $w_{ij}\leftarrow w_{ij}+\eta\,\Delta w_{ij}$ with $\Delta w_{ij}=\delta_jx_i$ ($w_{ij}$: weight from $i$ to
   $j$; $\eta$: step size).

*In words:* forward to compute, backward to assign blame; the gradient of every weight costs one forward and one
backward pass. With the cross-entropy loss, step 2 becomes $\delta_k=t_k-o_k$.

**Worked example.** $o_k=0.6$, $t_k=1$: squared error $\delta_k=0.6\cdot0.4\cdot0.4=0.096$; cross-entropy $\delta_k=0.4$.

> 🎓 **Exam tip.** Derive $\delta_k$ and $\delta_h$ yourself from $E=\frac12\sum_k(t_k-o_k)^2$ and $\sigma'=\sigma(1-\sigma)$;
remember the sign convention ($\delta=-\partial E/\partial\mathrm{net}$, hence "$+\eta$"). Know how many updates per
epoch each variant makes ($1$, $N$, $\lceil N/B\rceil$).

---
## 6 · Overfitting, capacity and the shape of the loss surface

> 📍 **Where we are.** A network with enough hidden units can fit almost any training set (highly nonlinear,
> potentially millions of parameters). With few samples that is a liability: the bias–variance trade-off of
> [Module 07](07_ridge_lasso_sparse_regression.ipynb) and [Module 23](23_model_selection.ipynb). Here we see it with
> only **40** training individuals and compare three remedies.

* **Capacity** — the number of hidden units sets how wiggly the function can be (for neural networks, the architecture
  is the model-selection knob; [Module 23](23_model_selection.ipynb)).
* **Early stopping** — stop gradient descent when the loss on a *validation* set (not the test set) stops improving.
* **Weight decay** — minimise $L(w)+\frac{\lambda}{2}\sum w^2$, which adds $\lambda w$ to every gradient and shrinks weights
  towards zero: ridge regression for networks, MAP with a Gaussian prior.

In [ ]:
# ── 6.1 Few samples, many hidden units: capacity, weight decay and early stopping ──
rng = np.random.default_rng(SEED + 9)
X_small, y_small = simulate_epistasis(40, rng)
X_val, y_val = simulate_epistasis(500, rng)            # validation set: used for early stopping only
T_small, T_val = y_small[:, None].astype(float), y_val[:, None].astype(float)
T_te = y_te[:, None].astype(float)


def report(label, p):
    """Print training accuracy, test accuracy and test cross-entropy of a fitted network."""
    print(f"  {label:32s} |   {accuracy(y_small, forward(p, X_small)[1][:, 0] > 0.5):.3f}     "
          f"{accuracy(y_te, forward(p, X_te)[1][:, 0] > 0.5):.3f}       {ce_loss(p, X_te, T_te):.3f}")


models = {}
print(f"  {'model (40 training samples)':32s} | train acc  test acc  test cross-entropy")
for n_h, wd in [(1, 0.0), (3, 0.0), (50, 0.0), (50, 1e-3), (50, 1e-2)]:
    p = init_params(2, n_h, 1, np.random.default_rng(SEED + 90), scale=2.0)
    h = train(p, X_small, T_small, lr=1.0, epochs=6000, weight_decay=wd, X_val=X_val, T_val=T_val)
    models[(n_h, wd)] = (p, h)
    report(f"{n_h:2d} hidden, weight decay {wd:g}", p)
best = int(np.argmin(models[(50, 0.0)][1]["val"]))           # epoch with the lowest validation loss
p_early = init_params(2, 50, 1, np.random.default_rng(SEED + 90), scale=2.0)
train(p_early, X_small, T_small, lr=1.0, epochs=best + 1)      # same start, stopped at that epoch
report(f"50 hidden, early stop @{best + 1}", p_early)
print(f"  {'guessing 0.5 for everyone':32s} |     –       0.500       {np.log(2):.3f}")
print(f"  {'Bayes-optimal classifier':32s} |     –       {accuracy(y_te, bayes_posterior(X_te) > 0.5):.3f}")

In [ ]:
# ── 6.2 Pictures of overfitting and of the two remedies ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, key, title in [(axes[0], (50, 0.0), "50 hidden units, no decay"), (axes[1], (50, 1e-3), "50 hidden units, decay 1e-3")]:
    p = models[key][0]
    plot_decision_regions(lambda P: (forward(p, P)[1][:, 0] > 0.5).astype(int), X_small, y_small, ax=ax, pad=0.8)
    ax.set_title(title + " (SIMULATED)"); ax.set_xlabel("allele signal, locus A"); ax.set_ylabel("allele signal, locus B")
h = models[(50, 0.0)][1]
axes[2].plot(h["train"], color="tab:blue", label="training loss (40 samples)")
axes[2].plot(h["val"], color="tab:orange", label="validation loss (500 samples)")
axes[2].axvline(best, color="k", ls=":", lw=1, label=f"early stop at epoch {best + 1}")
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("cross-entropy"); axes[2].set_ylim(0, 1.2)
axes[2].set_title("training keeps improving, validation turns up"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"validation loss: minimum {min(h['val']):.3f} after {best + 1} epochs, final {h['val'][-1]:.3f} after 6,000")

🔍 **What just happened?** Read the table row by row.

* **1 hidden unit** is just a slightly fancier logistic regression: it **underfits** (0.75 training, 0.60 test accuracy).
* **3 and 50 hidden units without a penalty** classify all 40 training individuals correctly — and reach only 0.75–0.79
  on new ones. Worse, their test **cross-entropy** (1.1–1.2) is higher than that of simply guessing 0.5 for everyone
  ($\log 2=0.693$): they are not just wrong sometimes, they are *confidently* wrong. This is **overfitting**; the left panel
  shows the boundary contorting itself around individual noisy training points. Accuracy alone hides it; the
  likelihood-based loss reveals it.
* **Weight decay 0.001** keeps the same 50 units but forces small weights: a smoother boundary (middle panel), 0.80 test
  accuracy and a test cross-entropy of 0.44. **Too much decay (0.01)** shrinks the weights so hard that the network can
  no longer express XOR and collapses to guessing — the bias side of the bias–variance trade-off.
* **Early stopping** gives the best result here (0.82 accuracy, cross-entropy 0.37): the right panel shows its signature —
  training loss decreases forever, validation loss reaches a minimum after about 140 epochs and then climbs; we simply
  keep the weights from the minimum.

No remedy can beat the Bayes-optimal accuracy (0.85): with 40 samples we get close, but not all the way.

⚠️ **The validation set is not the test set.** Early stopping and the choice of λ are *model selection*; if you choose
them on the test set, the test accuracy is no longer an honest estimate. [Module 23](23_model_selection.ipynb) is
devoted to doing this correctly (cross-validation, nested cross-validation).

In [ ]:
# ── 6.3 The loss surface is not convex: 20 random starts with 2 hidden units ──
final_losses, test_accs = [], []
for s in range(20):
    r = np.random.default_rng(1000 + s)
    p = init_params(2, 2, 1, r)
    final_losses.append(train(p, X_tr, T_tr, lr=2.0, epochs=2000)["train"][-1])
    test_accs.append(accuracy(y_te, forward(p, X_te)[1][:, 0] > 0.5))
final_losses, test_accs = np.array(final_losses), np.array(test_accs)
order = np.argsort(final_losses)
print("final training loss of each start (sorted):", np.round(final_losses[order], 3))
print("matching test accuracies               :", np.round(test_accs[order], 2))
print(f"starts that found the best 2-unit solution (test acc > 0.75): {np.sum(test_accs > 0.75)} of 20")

🔍 **What just happened?** Logistic regression has a convex loss: every start reaches the same optimum (Module 09).
A network does not. With only two hidden units — the bare minimum for XOR — the 20 random initialisations end in
**different places**. Thirteen reach a training loss of about 0.46 and a test accuracy of 0.77–0.79: the best a
2-unit network can do here, a diagonal *band* like the hand-built network of Section 2 (two units cannot draw the
"cross" that the Bayes-optimal rule, 0.85, needs). Seven get stuck at a higher loss (0.50–0.52) and about 0.65 test
accuracy: a **local minimum** in which the band covers only one of the two single-mutant clusters. In practice we
use more hidden units than strictly necessary (which makes bad minima rarer), random restarts, momentum-based
optimisers such as Adam, and a validation set to pick the best run.

✏️ **Your turn.** Repeat 6.3 with 6 hidden units instead of 2 (and the threshold 0.75 replaced by 0.83). How many of the
20 starts find a solution close to the Bayes-optimal accuracy now?

---
## 7 · The same network in PyTorch: automatic differentiation

> 📍 **Where we are.** You have written forward and backward passes by hand. Deep-learning libraries do the backward
> pass for you: while computing the forward pass they record every operation, and `loss.backward()` applies the chain
> rule through that record in reverse — **automatic differentiation**, i.e. backpropagation (Section 4) done by the computer.

Three PyTorch building blocks: a **tensor** is a NumPy-like array that can remember how it was computed;
`nn.Linear(n_in, n_out)` holds a weight matrix and an intercept vector and computes $XW^\top+b$; an **optimiser** such as
`torch.optim.SGD` applies the update $w\leftarrow w-\eta\,\nabla L$. We first check that PyTorch's gradients equal ours,
weight for weight, and then that training follows exactly the same trajectory.

In [ ]:
# ── 7.1 Copy our NumPy weights into PyTorch and compare gradients ──
def to_torch_net(params):
    """A PyTorch 2-n_h-1 sigmoid network (float64) holding the same weights as our NumPy dict."""
    n_in, n_h = params["W_h"].shape
    net = nn.Sequential(nn.Linear(n_in, n_h), nn.Sigmoid(), nn.Linear(n_h, 1)).double()
    with torch.no_grad():                                    # nn.Linear stores weights as (out, in): transpose
        net[0].weight.copy_(torch.from_numpy(params["W_h"].T)); net[0].bias.copy_(torch.from_numpy(params["b_h"]))
        net[2].weight.copy_(torch.from_numpy(params["W_k"].T)); net[2].bias.copy_(torch.from_numpy(params["b_k"]))
    return net


rng = np.random.default_rng(SEED + 10)
p = init_params(2, 4, 1, rng)
tnet = to_torch_net(p)
Xt, Tt = torch.from_numpy(X_tr), torch.from_numpy(T_tr)
loss_t = F.binary_cross_entropy_with_logits(tnet(Xt), Tt)   # sigmoid + cross-entropy in one stable step
loss_t.backward()                                            # automatic backpropagation
g_np = grads(p, X_tr, T_tr, "ce")
pairs = [(tnet[0].weight.grad.numpy().T, g_np["W_h"]), (tnet[0].bias.grad.numpy(), g_np["b_h"]),
         (tnet[2].weight.grad.numpy().T, g_np["W_k"]), (tnet[2].bias.grad.numpy(), g_np["b_k"])]
print(f"loss: NumPy {ce_loss(p, X_tr, T_tr):.12f} | PyTorch {loss_t.item():.12f}")
print("largest |autograd − our backprop| over all gradients: %.1e" % max(np.max(np.abs(a - b)) for a, b in pairs))

In [ ]:
# ── 7.2 Train with torch.optim.SGD from the same start: identical trajectory ──
p_np = init_params(2, 4, 1, np.random.default_rng(SEED + 11))
tnet = to_torch_net(p_np)                                    # copy BEFORE training the NumPy version
h_np = train(p_np, X_tr, T_tr, lr=2.0, epochs=500)["train"]
opt = torch.optim.SGD(tnet.parameters(), lr=2.0)
h_t = []
for epoch in range(500):
    loss_t = F.binary_cross_entropy_with_logits(tnet(Xt), Tt)
    opt.zero_grad()                                          # gradients accumulate unless reset
    loss_t.backward()
    opt.step()                                               # w ← w − lr · grad
    h_t.append(F.binary_cross_entropy_with_logits(tnet(Xt), Tt).item())
print(f"after 500 epochs: NumPy loss {h_np[-1]:.10f} | PyTorch loss {h_t[-1]:.10f}")
print("largest difference along the whole trajectory: %.1e" % np.max(np.abs(np.array(h_np) - np.array(h_t))))

🔍 **What just happened?** PyTorch's automatically computed gradients agree with our hand-derived backpropagation to
about $10^{-16}$ — the limit of double-precision arithmetic — and 500 epochs of training follow the same trajectory in
both implementations. There is no magic in `loss.backward()`: it is the backpropagation algorithm of Section 4. From now on we let
PyTorch differentiate, which frees us to try architectures whose gradients would be tedious to derive by hand.

⚠️ **Three PyTorch pitfalls.** (1) `nn.Linear` stores its weight as *(out_features, in_features)* — the transpose of
our $W_h$. (2) Gradients **accumulate** across `backward()` calls: call `opt.zero_grad()` every step. (3) Use
`binary_cross_entropy_with_logits` on the *pre-sigmoid* output rather than `sigmoid` followed by a log, which can
produce `log(0)` for confident predictions.

✏️ **Your turn.** Replace `nn.Sigmoid()` by `nn.ReLU()` and `torch.optim.SGD` by `torch.optim.Adam(tnet.parameters(),
lr=0.05)` in 7.2 (the NumPy comparison will no longer match, of course). Does the network reach a low loss in fewer epochs?

---
## 8 · Convolutional neural networks: scanning DNA for a motif

> 📍 **Where we are.** A fully connected network gives every input its own weight. For images and sequences that is
> wasteful: a cat is a cat wherever it appears, and a binding site is a binding site wherever it lies in the promoter.
> **Convolutional networks** build that knowledge into the architecture.

**Convolution on images.** In a single-layer convolutional network a small **local receptive field** (filter, e.g.
5 × 5 pixels of a 29 × 29 image of a handwritten digit) takes one patch of the image as input; there is one receptive
field (node) for each patch — a sliding window over the image — and the weights are **tied** together across all
receptive fields, so one filter extracts one feature everywhere. Multi-layer CNNs stack such layers; deep CNNs such as
LeNet-5 interleave convolution layers with **subsampling (pooling)** layers that take the maximum or average over small
neighbourhoods.

**Convolution on DNA.** For predicting transcription-factor binding sites the convolution is 1-D instead of 2-D. A DNA sequence of length $L$ becomes an $L\times4$ **one-hot** matrix (columns A, C,
G, T; one 1 per row) — a one-pixel-high image with four colour channels — and the filter is a $K\times4$ weight matrix
moved along the sequence:

$$s_p=b+\sum_{u=0}^{K-1}\ \sum_{c\in\{A,C,G,T\}}w_{u,c}\,x_{p+u,\,c}\qquad(p=0,1,\dots,L-K).$$

*In words:* at each position $p$, lay the filter over the next $K$ bases and add up the weights of the bases actually
present. If $w_{u,c}=\log_2\big(\mathrm{PWM}_{u,c}/0.25\big)$, where the **position weight matrix** (PWM) gives the
probability of base $c$ at motif position $u$, then $s_p$ is exactly the classical PWM log-odds score of the window
starting at $p$ (Durbin *et al.* 1998). **A convolution filter is a learnable PWM.**

In [ ]:
# ── 8.1 One-hot encoding and a PWM-shaped filter for the E-box CACGTG ──
BASES, EBOX, FIDELITY = "ACGT", "CACGTG", 0.9
PWM = np.full((len(EBOX), 4), (1 - FIDELITY) / 3)              # each position: consensus base with prob 0.9
for u, base in enumerate(EBOX):
    PWM[u, BASES.index(base)] = FIDELITY
W_ebox = np.log2(PWM / 0.25)                                    # log-odds weights (bits): a PWM-scoring filter
print(one_hot_dna("CACGTGA").astype(int), "← one_hot_dna('CACGTGA'), columns A C G T")
print("E-box filter weights (rows = motif positions, columns = A C G T):\n", np.round(W_ebox, 2))
revcomp = EBOX[::-1].translate(str.maketrans("ACGT", "TGCA"))
print(f"reverse complement of {EBOX} is {revcomp}: the E-box is a palindrome, so one filter covers both strands")

In [ ]:
# ── 8.2 Scan a sequence: convolution = PWM scoring at every position ──
def conv1d_dna(onehot, W, bias=0.0):
    """Move a K × 4 filter W along an L × 4 one-hot sequence; returns the L-K+1 window scores (cross-correlation)."""
    windows = np.lib.stride_tricks.sliding_window_view(onehot, W.shape)[:, 0]   # (L-K+1, K, 4)
    return bias + np.einsum("pkc,kc->p", windows, W)


rng = np.random.default_rng(SEED + 12)
seq = list(rng.choice(list(BASES), 60))
seq[27:33] = list(EBOX)                                          # plant one E-box at position 27
seq = "".join(seq)
scores = conv1d_dna(one_hot_dna(seq), W_ebox)
torch_scores = F.conv1d(torch.tensor(one_hot_dna(seq).T[None], dtype=torch.float64),
                        torch.tensor(W_ebox.T[None]))[0, 0].numpy()
print(seq)
print(f"best window starts at position {scores.argmax()} with score {scores.max():.2f} bits "
      f"(maximum possible {W_ebox.max(1).sum():.2f}); PyTorch conv1d agrees: {np.allclose(scores, torch_scores)}")

fig, (a0, ax) = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.3]))
a0.imshow(one_hot_dna(seq).T, aspect="auto", cmap="Greys", extent=(-0.5, len(seq) - 0.5, 3.5, -0.5))
a0.set_yticks(range(4)); a0.set_yticklabels(list(BASES)); a0.set_ylim(3.5, -1.8)
for start, col, lab in [(8, "tab:blue", "filter at p = 8"), (int(scores.argmax()), "tab:red", f"filter at p = {scores.argmax()}")]:
    a0.add_patch(plt.Rectangle((start - 0.5, -0.5), len(EBOX), 4, fill=False, ec=col, lw=2))
    a0.text(start + len(EBOX) / 2 - 0.5, -0.9, lab + " (same weights W)", color=col, ha="center", fontsize=8)
a0.set_title("one-hot DNA (black = base present): one 6 × 4 filter moves along the sequence — tied weights", fontsize=9)
ax.bar(np.arange(len(scores)), scores, color=np.where(scores == scores.max(), "tab:red", "tab:blue"))
ax.set_xlabel("window start position p (bp)"); ax.set_ylabel("filter score $s_p$ (bits)")
ax.set_title("the E-box filter scanned along a 60-bp SIMULATED sequence: one clear peak at the planted site", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The filter's score is low (strongly negative) almost everywhere and peaks at exactly the
planted position, where all six bases match the consensus (11.09 bits, the maximum). PyTorch's `conv1d` gives the same
numbers: what deep-learning libraries call "convolution" is, strictly speaking, **cross-correlation** — the filter is
not flipped — which does not matter because the filter is learned anyway.

Now the real test. We simulate 3,000 sequences of 60 bp. Half (the "bound" class) contain one E-box sampled from the
PWM — each position is the consensus base with probability 0.9, so most planted sites carry a mismatch or two, as real
binding sites do — at a random position; the other half are random. Can a CNN that has **never been told** the motif
learn to tell them apart, and does its filter recover `CACGTG`?

In [ ]:
# ── 8.3 SIMULATED 'bound' and 'unbound' sequences, and the Bayes-optimal benchmark ──
def simulate_bound_sequences(n, length, rng):
    """Half the sequences get one motif instance sampled from PWM at a random position (label 1), half are random."""
    seqs, labels = [], rng.permutation(np.r_[np.ones(n // 2, int), np.zeros(n - n // 2, int)])
    for lab in labels:
        s = rng.choice(list(BASES), length)
        if lab:
            pos = rng.integers(0, length - len(EBOX) + 1)
            s[pos:pos + len(EBOX)] = [BASES[rng.choice(4, p=row)] for row in PWM]
        seqs.append("".join(s))
    return seqs, labels


rng = np.random.default_rng(SEED + 13)
L_SEQ = 60
seqs, y_seq = simulate_bound_sequences(3000, L_SEQ, rng)
X_seq = np.stack([one_hot_dna(s).T for s in seqs]).astype(np.float32)        # (N, 4 channels, L) for PyTorch
n_train = 2000
print("data:", X_seq.shape, "| fraction bound:", y_seq.mean())
print("exact CACGTG among bound / unbound sequences:",
      np.mean([EBOX in s for s, l in zip(seqs, y_seq) if l == 1]).round(3), "/",
      np.mean([EBOX in s for s, l in zip(seqs, y_seq) if l == 0]).round(3))
# Bayes-optimal rule: likelihood ratio = average over positions of 2^(PWM score) (motif somewhere vs nowhere)
S_test = np.stack([conv1d_dna(one_hot_dna(s), W_ebox) for s in seqs[n_train:]])
bayes_pred = (np.mean(2.0 ** S_test, axis=1) > 1).astype(int)
print("Bayes-optimal test accuracy (knows the true PWM):", round(accuracy(y_seq[n_train:], bayes_pred), 3))

🔍 **What just happened?** Only about half of the "bound" sequences contain a perfect `CACGTG` (because each of the six
positions is correct with probability 0.9, and $0.9^6\approx0.53$), and about 1 % of the random sequences contain one
by chance (55 windows × $4^{-6}$ ≈ 1.3 %; 0.013 printed above). So the classes genuinely overlap. Because we simulated the data, we can
compute the **Bayes-optimal** classifier: the likelihood ratio "motif somewhere" versus "no motif" equals the average
over positions of $2^{s_p}$ with $s_p$ the PWM score. Its accuracy is the ceiling for any method on this problem.

Our CNN: one convolution layer with 16 filters of width 8 (a little wider than the motif, so it need not be told the
motif length), a ReLU, **global max pooling** ("does this filter fire strongly *anywhere*?" — the max-pooling step of
the binding-site architecture described at the start of this notebook),
and one output neuron combining the 16 maxima.

In [ ]:
# ── 8.4 A 1-D CNN motif detector, trained with Adam ──
class MotifCNN(nn.Module):
    """conv1d(4 → n_filters, width k) → ReLU → global max pool over positions → linear → logit of P(bound)."""

    def __init__(self, n_filters=16, k=8):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, kernel_size=k)
        self.out = nn.Linear(n_filters, 1)

    def forward(self, x):
        h = F.relu(self.conv(x))                 # (N, n_filters, L-k+1): one score track per filter
        return self.out(h.max(dim=2).values).squeeze(1)


def fit_classifier(model, X, y, n_train, epochs=20, lr=3e-3, batch=64, seed=SEED):
    """Mini-batch Adam on the first n_train rows; returns (train accuracy, test accuracy, seconds)."""
    torch.manual_seed(seed)
    Xt, yt = torch.from_numpy(X), torch.tensor(y, dtype=torch.float32)
    opt, gen, t0 = torch.optim.Adam(model.parameters(), lr=lr), torch.Generator().manual_seed(seed), time.time()
    for _ in range(epochs):
        perm = torch.randperm(n_train, generator=gen)
        for s in range(0, n_train, batch):
            idx = perm[s:s + batch]
            loss = F.binary_cross_entropy_with_logits(model(Xt[idx]), yt[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pred = (model(Xt) > 0).numpy().astype(int)
    return accuracy(y[:n_train], pred[:n_train]), accuracy(y[n_train:], pred[n_train:]), time.time() - t0


torch.manual_seed(SEED)
cnn = MotifCNN()
acc_tr, acc_te, secs = fit_classifier(cnn, X_seq, y_seq, n_train)
print(f"CNN: {sum(p.numel() for p in cnn.parameters())} parameters | train acc {acc_tr:.3f} | "
      f"test acc {acc_te:.3f} | {secs:.1f} s on CPU")

In [ ]:
# ── 8.5 Open the black box: what did the most useful filter learn? ──
W_learned = cnn.conv.weight.detach().numpy()                     # (16 filters, 4 bases, 8 positions)
f_best = int(np.argmax(cnn.out.weight.detach().numpy()[0]))      # filter with the largest positive output weight
consensus = "".join(BASES[i] for i in W_learned[f_best].argmax(axis=0))
print("output weights of the 16 filters:", np.round(cnn.out.weight.detach().numpy()[0], 2))
print(f"filter {f_best}: highest-weight base at each of its 8 positions → {consensus}"
      f"   (contains {EBOX}: {EBOX in consensus})")

fig, axes = plt.subplots(1, 2, figsize=(11, 2.8), gridspec_kw={"width_ratios": [8, 6]})
for ax, Wm, title in [(axes[0], W_learned[f_best], f"LEARNED filter {f_best} (from SIMULATED data)"),
                      (axes[1], W_ebox.T, "true PWM log-odds (used to simulate)")]:
    v = np.abs(Wm).max()
    ax.imshow(Wm, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
    ax.set_yticks(range(4)); ax.set_yticklabels(list(BASES)); ax.set_xlabel("position in filter")
    for u in range(Wm.shape[1]):
        ax.text(u, Wm[:, u].argmax(), BASES[Wm[:, u].argmax()], ha="center", va="center", fontsize=11, weight="bold")
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In a few seconds of CPU time the CNN (545 parameters) reaches a test accuracy of 0.82,
less than two percentage points below the Bayes-optimal ceiling of 0.84 computed in 8.3 — it has learned almost
everything there is to learn. Opening it up, the filter with the largest positive output weight has the E-box written
into its weights: its highest-scoring base at each position spells `CACGTGGG`, i.e. `CACGTG` followed by two positions
with weak, nearly uninformative weights (the filter is 8 wide, the motif 6). Other filters with positive output weights
typically hold shifted or partial copies of the motif; filters with negative weights learn weaker "anti-motif" patterns. The
network **rediscovered the motif from labelled sequences alone** — exactly what DeepBind-style models do with real
ChIP-seq data, where the "answer" is not known in advance.

⚠️ **Interpretation is a hypothesis, not a proof.** On real data a filter that resembles a known motif should be
compared with motif databases (e.g. JASPAR) and tested experimentally; filters can also learn technical artefacts
(GC content, repeats, sequencing biases) that happen to correlate with the label.

In [ ]:
# ── 8.6 Why weight tying matters: a fully connected network on the same data ──
torch.manual_seed(SEED)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(4 * L_SEQ, 32), nn.ReLU(), nn.Linear(32, 1), nn.Flatten(0))
acc_tr_mlp, acc_te_mlp, secs_mlp = fit_classifier(mlp, X_seq, y_seq, n_train)
print(f"fully connected MLP: {sum(p.numel() for p in mlp.parameters()):,} parameters | train acc {acc_tr_mlp:.3f} | "
      f"test acc {acc_te_mlp:.3f}")
print(f"CNN                : {sum(p.numel() for p in cnn.parameters()):,} parameters | train acc {acc_tr:.3f} | "
      f"test acc {acc_te:.3f}")

🔍 **What just happened?** The fully connected network has about 14 times more parameters (7,745 vs 545), fits the
2,000 training sequences essentially perfectly (0.999) — and is no better than a coin flip on new sequences (0.49). It has no notion that a
motif at position 5 and the same motif at position 40 mean the same thing, so it would have to learn the motif
separately at each of the 55 possible positions from roughly 1,000 bound examples (≈ 18 per position) — instead, it
memorises. The CNN's **tied weights** encode the biological prior "a binding site is the same pattern wherever it
occurs", and global max pooling makes the prediction **translation invariant**. A good architecture is a model
assumption, just like conditional independence in a Bayesian network.

✏️ **Your turn.** (a) Change `FIDELITY` in cell 8.1 to 0.75 and rerun cells 8.1–8.6: what happens to the Bayes-optimal
accuracy, the CNN, and the learned filter? (b) Make the filters too short (`MotifCNN(k=4)`): can a 4-wide filter still
detect a 6-bp motif? (c) Give the MLP 10× more training sequences — does it catch up?

### 📘 Lecture notes — convolutional neural networks

**Feed-forward vs convolutional.** A fully connected feed-forward network gives every input its own weight. A
convolutional network uses a **local receptive field** (filter) that takes one patch of the input; there is one
receptive field for each patch (a sliding window), and the **weights are tied** across all receptive fields, so the
filter extracts the same feature everywhere. Several filters give several feature maps; multi-layer CNNs stack
convolution layers, and deep CNNs (LeNet-5) interleave them with **subsampling** layers (max or average pooling).

**1-D CNNs for transcription-factor binding sites.** Input: the one-hot DNA sequence ($L\times4$), possibly together
with its reverse complement; convolution (1-D instead of 2-D) → ReLU → max pooling → fully connected layers →
prediction, trained against references from functional experiments (ChIP-seq). Filter score at position $p$:
$$s_p=b+\sum_{u=0}^{K-1}\sum_{c}w_{u,c}\,x_{p+u,c}$$
— with $w_{u,c}=\log_2(\mathrm{PWM}_{u,c}/0.25)$ exactly the position-weight-matrix log-odds score, so **a filter is a
learnable PWM**.

**Why it works.** Tied weights encode "a binding site is the same pattern wherever it occurs" (translation
equivariance) and slash the number of parameters ($n_{\text{filters}}\times4\times K+n_{\text{filters}}$, independent of
$L$); global max pooling makes the prediction translation invariant. In Section 8 a 545-parameter CNN nearly reached
the Bayes-optimal accuracy while a 7,745-parameter fully connected network memorised the training set.

> 🎓 **Exam tip.** Be able to count the parameters of a convolution layer ($n_{\text{filters}}\times n_{\text{channels}}\times k$
weights + $n_{\text{filters}}$ biases; independent of sequence length!) and of a fully connected layer on the same
input, and to explain what weight tying and max pooling buy (translation equivariance / invariance).

---
## 9 · Graph neural networks: convolution over a network of molecules

> 📍 **Where we are.** A CNN's filter looks at a node's neighbours on a regular grid (adjacent pixels or bases). Many
> biological data live on **irregular graphs**: protein–protein interaction networks, drug–target networks, molecules
> (atoms and bonds). **Graph convolutional networks** apply the convolutional filter over a graph instead of an image:
> the "filter" moves over each node's neighbourhood, and networks are the input instead of images.

A landmark example is **Decagon** (Zitnik, Agrawal & Leskovec 2018), which predicts *polypharmacy side effects* — side
effects that appear when two drugs are taken together, such as gastrointestinal bleeding or bradycardia — from a graph
of drugs (e.g. ciprofloxacin, doxycycline, simvastatin, mupirocin) and proteins, with drug–target and protein–protein
edges and a feature vector at every node. Each node's new representation is computed from its neighbours' representations with weights
that are **shared by all nodes** — a single filter is defined for each node (across its neighbours), and the weights of
all filters for all nodes are tied together — just as a CNN filter is shared by all positions. A widely used simple form (Kipf & Welling 2017) is

$$H'=\mathrm{ReLU}\big(\hat A\,H\,W\big),\qquad \hat A=\tilde D^{-1/2}(A+I)\,\tilde D^{-1/2}.$$

*In words:* $H$ has one row of features per node; $A$ is the **adjacency matrix** ($A_{uv}=1$ if nodes $u$ and $v$ are
linked); adding the identity $I$ lets each node keep its own features; $\tilde D$ is the diagonal matrix of node degrees
of $A+I$, and the $\tilde D^{-1/2}$ factors turn the sum over neighbours into a degree-normalised average. Multiplying by
$\hat A$ **mixes each node's features with its neighbours'** ("message passing"); multiplying by $W$ applies the same
learned transformation to every node; ReLU adds the nonlinearity. Decagon uses a separate $W_r$ for each edge type $r$.

In [ ]:
# ── 9.1 A tiny SIMULATED drug–protein graph and one graph-convolution layer ──
nodes = ["Dox", "Cip", "Sim", "Mup", "P1", "P2", "P3", "P4", "P5"]          # 4 drugs, 5 proteins (illustrative)
edges = [("Dox", "P1"), ("Cip", "P1"), ("Cip", "P2"), ("Cip", "P3"), ("Sim", "P4"), ("Mup", "P5"),   # drug–target
         ("P1", "P2"), ("P2", "P3"), ("P3", "P4"), ("P4", "P5")]                                       # protein–protein
idx = {v: i for i, v in enumerate(nodes)}
A = np.zeros((len(nodes), len(nodes)))
for u, v in edges:
    A[idx[u], idx[v]] = A[idx[v], idx[u]] = 1                                 # undirected
A_tilde = A + np.eye(len(nodes))
d_inv_sqrt = 1 / np.sqrt(A_tilde.sum(axis=1))
A_hat = d_inv_sqrt[:, None] * A_tilde * d_inv_sqrt[None, :]

rng = np.random.default_rng(SEED + 17)
H = np.c_[np.r_[np.ones(4), np.zeros(5)], rng.normal(size=(9, 2))]          # features: is-drug flag + 2 random descriptors
W_gcn = rng.normal(size=(3, 4))                                              # ONE weight matrix shared by all nodes
H1 = np.maximum(0, A_hat @ H @ W_gcn)
print("H", H.shape, "→ H' = ReLU(Â H W)", H1.shape)
reach1 = [nodes[j] for j in np.flatnonzero(A_hat[idx["Cip"]])]
reach2 = [nodes[j] for j in np.flatnonzero((A_hat @ A_hat)[idx["Cip"]])]
print("nodes that influence Cip's representation after 1 layer :", reach1)
print("                                            after 2 layers:", reach2)

fig, ax = plt.subplots(figsize=(6, 3.2))
pos = {"Dox": (0.05, 0.85), "Cip": (0.35, 0.9), "Sim": (0.75, 0.9), "Mup": (0.98, 0.6),
       "P1": (0.15, 0.4), "P2": (0.4, 0.15), "P3": (0.6, 0.45), "P4": (0.8, 0.2), "P5": (1.0, 0.05)}
draw_graph(edges, pos, directed=False, ax=ax, highlight={"Dox", "Cip", "Sim", "Mup"},
           title="SIMULATED drug (orange) – protein (blue) graph")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.2 Tied weights ⇒ relabelling the nodes just relabels the outputs ──
perm = np.random.default_rng(SEED + 15).permutation(len(nodes))
P = np.eye(len(nodes))[perm]                                       # permutation matrix: reorders the nodes
H1_perm = np.maximum(0, (P @ A_hat @ P.T) @ (P @ H) @ W_gcn)
print("GCN output for the relabelled graph equals the relabelled output:", np.allclose(H1_perm, P @ H1))
z = H1 / np.linalg.norm(H1, axis=1, keepdims=True).clip(1e-9)      # node embeddings (untrained!)
for a, b in [("Cip", "Mup"), ("Cip", "Sim"), ("Dox", "Cip")]:
    print(f"untrained pair score sigma(z_{a} · z_{b}) = {sigmoid(z[idx[a]] @ z[idx[b]]):.3f}")

🔍 **What just happened?** After one layer, Ciprofloxacin's representation depends only on itself and its direct
neighbours (its three target proteins); after two layers, on everything within two steps — the graph analogue of a CNN's
receptive field growing with depth. Relabelling the nodes simply relabels the outputs (**permutation equivariance**), a
direct consequence of the tied weights: the layer cannot depend on arbitrary node numbering, only on the graph
structure and features. Finally, a *decoder* turns two node embeddings into a score for an interaction (here a simple
dot product; Decagon uses a learned, side-effect-specific bilinear form). These scores are meaningless until the
weights are trained — by backpropagation and SGD on known drug-pair side effects, exactly as in Sections 4–7.

⚠️ **Graphs are hypotheses too.** A predicted interaction is a statistical association learned from the graph and the
labels; like a learned Bayesian-network edge ([Module 17](17_bayesian_networks_inference_learning.ipynb)) it is not proof
of a mechanism.

✏️ **Your turn.** Add an edge ("Mup", "P3") and rerun 9.1. Which nodes now influence Cip after two layers?

### 📘 Lecture notes — graph convolutional networks

**From images to graphs.** Replace the grid of pixels by a graph (protein–protein interactions, drug–target
networks, molecules). A single filter is defined for each node across its neighbours, and the weights of all filters
for all nodes are tied together. One layer (Kipf & Welling form):
$$H'=\mathrm{ReLU}(\hat AHW),\qquad\hat A=\tilde D^{-1/2}(A+I)\tilde D^{-1/2}.$$
*In words:* average each node's features with its neighbours' (message passing), transform them with one shared
weight matrix $W$, apply a nonlinearity; $k$ layers see $k$-hop neighbourhoods.

**Drug interactions (Decagon).** Drugs and proteins as nodes, drug–protein and protein–protein edges, a separate
weight matrix per edge type; the learned node embeddings of two drugs are combined by a decoder into the probability
of each polypharmacy side effect ($r_1$: gastrointestinal bleed, $r_2$: bradycardia, …). Trained by backpropagation
and SGD on known side effects.

> 🎓 **Exam tip.** Know the analogy: image patch ↔ node neighbourhood, tied filter weights ↔ one $W$ shared by all nodes, deeper
layers ↔ larger receptive fields ($k$ layers see $k$-hop neighbours).

---
## 10 · Autoencoders, variational autoencoders and recurrent networks

> 📍 **Where we are.** So far every network was **supervised** (labels $Y$). Neural networks require a large number of
> training samples, and labelled biological samples are scarce while unlabelled ones are abundant. **Autoencoders**
> learn from abundant unlabelled data (unsupervised learning) — neural networks for feature extraction, "a fancier PCA"
> — and connect directly to [Module 14](14_pca_dimensionality_reduction.ipynb).

An **autoencoder** is trained to reproduce its input at its output, $\hat x=g(f(x))$, by minimising the reconstruction
error $\frac1N\sum_m\|x_m-\hat x_m\|^2$. The **encoder** $f$ maps the data to a low-dimensional **code** $z=f(x)$; the
**decoder** $g$ maps the code back (from the low-dimensional embedding to the data). The **bottleneck** (code dimension
$m$ much smaller than the number of genes $J$) is essential: it forces the model to learn a compressed (latent)
representation; without it the network could learn the identity function and nothing useful.

**Linear autoencoder = PCA.** If $f(x)=W_ex$ and $g(z)=W_dz$ are linear and the loss is the squared error, the best
possible reconstruction is the projection onto the top-$m$ principal components, and the decoder's columns span exactly
the PCA subspace (Baldi & Hornik 1989). Let us check this on the SIMULATED leukaemia data (shaped like Golub
*et al.* 1999; 47 ALL — 38 B-lineage, 9 T-lineage — and 25 AML patients).

In [ ]:
# ── 10.1 A linear autoencoder vs PCA on the SIMULATED leukaemia data ──
leuk = simulate_leukemia_expression()                          # fixed default seed: the course's standard data set
print("source:", leuk["source"], "| X:", leuk["X"].shape, "| AML:", leuk["y"].sum(), "ALL:", (leuk["y"] == 0).sum())
top = np.argsort(leuk["X"].var(axis=0))[::-1][:1000]          # the 1,000 most variable genes
Z_leuk = standardize(leuk["X"][:, top])[0]                    # z-score each gene (mean 0, sd 1)
U, S, Vt = np.linalg.svd(Z_leuk, full_matrices=False)          # PCA via the SVD (Module 14)
pca_mse = np.mean((Z_leuk - (U[:, :2] * S[:2]) @ Vt[:2]) ** 2)
print(f"PCA, 2 components: reconstruction MSE {pca_mse:.4f} (variance explained {np.sum(S[:2]**2)/np.sum(S**2):.1%})")

torch.manual_seed(SEED)
encoder, decoder = nn.Linear(1000, 2, bias=False), nn.Linear(2, 1000, bias=False)
opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)
Zt = torch.tensor(Z_leuk, dtype=torch.float32)
for it in range(2000):                                          # full-batch training on the 72 patients
    ae_loss = ((decoder(encoder(Zt)) - Zt) ** 2).mean()
    opt.zero_grad(); ae_loss.backward(); opt.step()
print(f"linear autoencoder, 2-unit bottleneck: reconstruction MSE {ae_loss.item():.4f}")
Q_ae = np.linalg.qr(decoder.weight.detach().numpy().astype(float))[0]     # orthonormal basis of the decoder's columns
cosines = np.linalg.svd(Q_ae.T @ Vt[:2].T, compute_uv=False)
print("cosines of the principal angles between the AE subspace and PC1–PC2:", np.round(cosines, 5))
print("fraction of variance explained by PC1, PC2, PC3:", np.round(S[:3] ** 2 / np.sum(S ** 2), 4))

In [ ]:
# ── 10.2 The two 2-D maps of the patients ──
codes = encoder(Zt).detach().numpy()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
draw_net(axes[0], [[""] * 6, [""] * 4, [""] * 2, [""] * 4, [""] * 6], r=0.18, dx=1.5, fs=7,
         colors=["#c6dbef", "#fdd0a2", "#fcbba1", "#fdd0a2", "#c6dbef"],
         layer_names=["input x", "encoder", "code z", "decoder", "output x̂"],
         title="an autoencoder: trained with the inputs as outputs,\nthrough a bottleneck (here 2 units)")
for ax, P2, title in [(axes[1], U[:, :2] * S[:2], "PCA scores (PC1, PC2)"), (axes[2], codes, "linear autoencoder codes")]:
    for c, lab in enumerate(["B-ALL", "T-ALL", "AML"]):
        m = leuk["subtype"] == lab
        ax.scatter(P2[m, 0], P2[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=lab)
    ax.set_title(title + "\n(SIMULATED leukaemia data)", fontsize=9); ax.set_xlabel("dimension 1"); ax.set_ylabel("dimension 2")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The linear autoencoder, trained by gradient descent with no knowledge of eigenvectors, reaches
the same reconstruction error as two principal components, and its decoder spans the same plane: both principal-angle
cosines are 1 to five decimals. (It needs a couple of thousand Adam steps here, because the second and third principal
components explain almost the same variance, 4.9 % and 4.7 %, so the loss is nearly flat between them.) Its 2-D codes
are a rotated and stretched copy of the PCA scores (the autoencoder has no reason to make its two axes orthogonal or
ordered by variance), so the structure looks the same up to a linear transformation: 🧪 three groups that coincide with
the three simulated subtypes — AML, B-ALL and T-ALL — although neither method was told any labels. A linear autoencoder
*is* PCA, found the slow way. Interesting autoencoders are **nonlinear** and
**deep** — more layers learn better encodings: a 2000–1000–500–30 encoder with a mirror-image decoder reconstructs
images of faces from a code of just 30 numbers, far better than 30-component PCA (Hinton & Salakhutdinov 2006) — but
they need data. How much? Let us try a nonlinear autoencoder on the 38 training patients of the original 38/34
training/test design and test it on the other 34.

In [ ]:
# ── 10.3 Nonlinear autoencoder with 38 patients: memorisation ──
tr = leuk["is_train"]
Z_tr, mu, sd = standardize(leuk["X"][tr][:, top])
Z_te = standardize(leuk["X"][~tr][:, top], mu, sd)[0]
A_tr, A_te = torch.tensor(Z_tr, dtype=torch.float32), torch.tensor(Z_te, dtype=torch.float32)
V2 = np.linalg.svd(Z_tr, full_matrices=False)[2][:2].T                # PCA on the 38 training patients
results = {"predict the mean (no model)": (np.mean(Z_tr ** 2), np.mean(Z_te ** 2)),
           "PCA, 2 components": (np.mean((Z_tr - Z_tr @ V2 @ V2.T) ** 2), np.mean((Z_te - Z_te @ V2 @ V2.T) ** 2))}
for name, layers in [("linear AE, bottleneck 2", [nn.Linear(1000, 2), nn.Linear(2, 1000)]),
                     ("nonlinear AE 1000-32-2-32-1000", [nn.Linear(1000, 32), nn.Tanh(), nn.Linear(32, 2),
                                                         nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 1000)])]:
    torch.manual_seed(SEED)
    ae = nn.Sequential(*layers)
    opt = torch.optim.Adam(ae.parameters(), lr=0.003)
    for it in range(1500):
        loss = ((ae(A_tr) - A_tr) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        results[name] = (loss.item(), ((ae(A_te) - A_te) ** 2).mean().item())
print(f"{'model':32s} train MSE  test MSE   (38 training / 34 test patients)")
for name, (a, b) in results.items():
    print(f"{name:32s}  {a:7.3f}   {b:7.3f}")

🔍 **What just happened?** On the training patients the nonlinear autoencoder looks spectacular: its reconstruction
error (0.09) is about a tenth of PCA's (0.85), through a bottleneck of just two numbers. On new patients it is a
disaster: 2.02, about twice the error of not modelling at all ("predict the mean", 1.07). With 38 samples and ~65,000
parameters it has simply memorised 38 patients (two code numbers are enough to index 38 stored profiles). PCA improves
modestly on the mean (1.00); the linear autoencoder matches PCA on the training patients (0.85) but hardly helps on new
ones (1.07): with 38 patients in 1,000 dimensions its encoder is not pinned down in directions that the training data
never visit, whereas PCA's projection is. Both simple models still generalise far better than the deep one. This is the warning of this section's introduction made concrete: **neural networks require a large number of training samples**; with
tens of patients, simpler models win. Single-cell RNA-seq, with hundreds of thousands of cells, is where deep
autoencoders shine.

### 10.4 Variational autoencoders: the probabilistic version of the autoencoder

A **variational autoencoder** (VAE; Kingma & Welling 2014) turns the decoder into a **generative model**: draw a latent
code $z\sim\mathcal N(0,I)$, then draw the data $x\sim p_\theta(x\mid z)$, whose mean is computed by a deep network
$g_\theta(z)$. The encoder becomes **inference**: a network that outputs an approximation $q_\phi(z\mid x)$ (a Gaussian
with mean $\mu_\phi(x)$ and variance $\sigma^2_\phi(x)$) to the posterior $p(z\mid x)$. Both are trained together by
maximising a lower bound on the log-likelihood — "maximise the likelihood of the data, plus good inference":

$$\log p_\theta(x)\;\ge\;\mathbb E_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]-\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big).$$

*In words:* the first term rewards codes from which the data can be reconstructed well (the autoencoder part); the
second, a Kullback–Leibler divergence, keeps the encoder's distribution close to the prior so that codes are smooth and
new samples can be generated. This is the course's **latent-variable** idea again: $z$ plays the role of the hidden
cluster label of a mixture model or the hidden state of an HMM, $q_\phi$ plays the role of the E-step's responsibilities
([Module 13](13_mixture_models_em.ipynb)), and with a *linear* decoder the model reduces to probabilistic PCA (Tipping &
Bishop 1999). scVI (Lopez *et al.* 2018) is a widely used VAE for single-cell RNA-seq. For one-dimensional Gaussians the KL
term has a closed form, which we verify by sampling:

In [ ]:
# ── 10.4 The VAE's KL term for q = N(mu, s^2) and prior p = N(0, 1): formula vs Monte Carlo ──
rng = np.random.default_rng(SEED + 16)
for mu_q, s_q in [(0.0, 1.0), (1.5, 0.5), (-2.0, 2.0)]:
    closed = 0.5 * (mu_q ** 2 + s_q ** 2 - 1 - np.log(s_q ** 2))
    zs = rng.normal(mu_q, s_q, 200_000)                                    # samples from q
    log_q = -0.5 * np.log(2 * np.pi * s_q ** 2) - (zs - mu_q) ** 2 / (2 * s_q ** 2)
    log_p = -0.5 * np.log(2 * np.pi) - zs ** 2 / 2
    print(f"q = N({mu_q:+.1f}, {s_q:.1f}²):  KL closed form {closed:.4f} | Monte Carlo E_q[log q − log p] {np.mean(log_q - log_p):.4f}")

🔍 **What just happened?** The KL divergence is zero when the encoder's distribution equals the prior and grows as its
mean moves away from 0 or its spread differs from 1; the Monte Carlo average of $\log q-\log p$ over 200,000 samples
matches the formula to two or three decimals. A VAE adds this penalty, computed for every training cell or patient, to
the reconstruction error.

### 10.5 Recurrent neural networks: networks with memory

For sequences and time series (text, DNA, video), a **recurrent** network keeps a hidden state $s_t$ that is updated as
each new input $x_t$ arrives:

$$s_t=f\big(Ux_t+Ws_{t-1}+b\big),\qquad o_t=g\big(Vs_t\big).$$

*In words:* the new state mixes the current input (through $U$) with the previous state (through $W$, the feedback
loop that turns a feed-forward network into a recurrent one); the output is read from the state (through $V$).
**Unrolling** the loop gives a deep
feed-forward network with one layer per time step and the **same** $U,W,V$ at every step — weight tying in time, just as
a CNN ties weights in space. Training uses backpropagation through the unrolled network ("backpropagation through
time"). Compare the HMM of [Module 19](19_hmm_inference.ipynb): it also has a chain of hidden states, but those are
discrete random variables with transition probabilities, whereas an RNN's state is a deterministic vector of numbers.

Here is a hand-wired RNN with two hidden units that counts **CpG dinucleotides** (a C followed by a G, the target of DNA
methylation): unit 1 remembers "the current base is C"; unit 2 fires when "the previous base was C (unit 1 at $t-1$) and
the current base is G".

In [ ]:
# ── 10.5 A hand-wired recurrent network that detects CpG ──
k = 20.0                                                       # large weights → nearly binary sigmoid units
U_rnn = np.array([[0, k, 0, 0],                                # unit 1 listens to 'C' (columns A C G T)
                  [0, 0, k, 0]])                               # unit 2 listens to 'G'
W_rnn = np.array([[0, 0],
                  [k, 0]])                                     # unit 2 also listens to unit 1 at the previous step
b_rnn = np.array([-k / 2, -1.5 * k])                           # unit 2 needs BOTH inputs (an AND gate, Section 1)
V_rnn = np.array([0.0, 1.0])                                   # output = unit 2


def run_rnn(seq):
    """Unroll the RNN along a DNA string; returns the output o_t at every position."""
    s, outputs = np.zeros(2), []
    for x_t in one_hot_dna(seq):                               # the SAME U, W, b, V at every time step
        s = sigmoid(U_rnn @ x_t + W_rnn @ s + b_rnn)
        outputs.append(V_rnn @ s)
    return np.array(outputs)


dna = "ATCGGCGTACGATTCCGGACGCGT"
o = run_rnn(dna)
print(dna)
print("".join("^" if v > 0.5 else " " for v in o), "← positions where the RNN fires")
print(f"RNN count of CpG: {int(np.sum(o > 0.5))} | direct count of 'CG': {sum(dna[i:i + 2] == 'CG' for i in range(len(dna) - 1))}")
rng = np.random.default_rng(SEED + 18)
long_dna = "".join(rng.choice(list(BASES), 5000))
print("on a random 5,000-bp sequence:", int(np.sum(run_rnn(long_dna) > 0.5)), "vs",
      sum(long_dna[i:i + 2] == "CG" for i in range(len(long_dna) - 1)))

fig, ax = plt.subplots(figsize=(11, 2.9))                     # the network folded (left) and unrolled in time (right)
for (x, y), lab, col in [((0, 0), "$x$", "#c6dbef"), ((0, 1.2), "$s$", "#fdd0a2"), ((0, 2.4), "$o$", "#fcbba1")]:
    ax.add_patch(plt.Circle((x, y), 0.25, fc=col, ec="k", zorder=3)); ax.text(x, y, lab, ha="center", va="center", zorder=4)
for y0, y1, lab in [(0.25, 0.95, "U"), (1.45, 2.15, "V")]:
    ax.annotate("", xy=(0, y1), xytext=(0, y0), arrowprops=dict(arrowstyle="-|>", lw=1))
    ax.text(0.12, (y0 + y1) / 2, lab, fontsize=9)
ax.annotate("", xy=(0.2, 1.35), xytext=(0.2, 1.05), zorder=5,
            arrowprops=dict(arrowstyle="-|>", lw=1, connectionstyle="arc3,rad=1.8", shrinkA=0, shrinkB=0))
ax.text(0.6, 1.15, "W", fontsize=9)
ax.text(1.5, 1.2, "unroll\n⟹", ha="center", va="center", fontsize=11)
s_t = np.zeros(2)
for t, base in enumerate(dna[:8]):                            # first 8 bases of the example, with the RNN's states
    s_t = sigmoid(U_rnn @ one_hot_dna(base)[0] + W_rnn @ s_t + b_rnn)
    X0 = 2.5 + 1.15 * t
    for y, lab, col in [(0, base, "#c6dbef"), (1.2, f"{s_t[0]:.0f},{s_t[1]:.0f}", "#fdd0a2"),
                        (2.4, "1" if s_t[1] > 0.5 else "0", "#fcbba1")]:
        ax.add_patch(plt.Circle((X0, y), 0.3, fc=col, ec="k", zorder=3))
        ax.text(X0, y, lab, ha="center", va="center", fontsize=8, zorder=4)
    for y0, y1 in [(0.3, 0.9), (1.5, 2.1)]:
        ax.annotate("", xy=(X0, y1), xytext=(X0, y0), arrowprops=dict(arrowstyle="-|>", lw=0.8))
    if t:
        ax.annotate("", xy=(X0 - 0.3, 1.2), xytext=(X0 - 0.85, 1.2), arrowprops=dict(arrowstyle="-|>", lw=0.8, color="tab:red"))
ax.text(X0 + 0.6, 0, "input $x_t$", va="center", fontsize=8); ax.text(X0 + 0.6, 1.2, "state $s_t$ (units 1, 2)", va="center", fontsize=8)
ax.text(X0 + 0.6, 2.4, "output $o_t$", va="center", fontsize=8)
ax.set_xlim(-0.5, X0 + 2.6); ax.set_ylim(-0.5, 2.9); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("a recurrent network folded (feedback loop W) and unrolled over the first 8 bases: the same U, W, V at every "
             "step (red: W)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The network fires exactly at the G of every CG, and its count matches a direct string count,
on the short example and on 5,000 random bases. Two hidden units and six nonzero numbers (two in $U$, one in $W$, two intercepts, one in $V$), reused at every position,
implement a little finite-state machine: **memory** through the recurrent weight $W$. A trained RNN (or its modern
successors, LSTMs and transformers) discovers such state machines itself from data, with far more units — but the
principle is the one you just ran by hand.

✏️ **Your turn.** Rewire the RNN to detect the dinucleotide **TA** instead (hint: change which columns of `U_rnn` carry
the weight `k`). Then try a trinucleotide such as **CAC** — how many hidden units do you need?

### 📘 Lecture notes — autoencoders, variational autoencoders and recurrent networks

**Autoencoder** — neural networks for feature extraction ("fancier PCA"), unsupervised, so they can learn from
abundant unlabelled data (networks need many training samples). Trained with the inputs as outputs: reconstruct
$x$ from itself through a **bottleneck** that forces a compressed (latent) code and prevents learning the identity
function. **Encoder:** data → low-dimensional embedding $z=f(x)$; **decoder:** embedding → data $\hat x=g(z)$; loss
$\frac1N\sum\|x-\hat x\|^2$. More layers learn better encodings (e.g. 2000–1000–500–30 for face images). A linear
autoencoder spans the PCA subspace.

**Variational autoencoder** — the probabilistic version. Decoder = generative model $p(x\mid z)p(z)$ with $z$ hidden
variables (features), $x$ observed data and $p(x\mid z)$ a deep network; encoder = inference on the generative model,
an approximation $q(z\mid x)$ to $p(z\mid x)$. Train encoder + decoder together by maximising the likelihood of the
data plus the quality of inference:
$$\log p(x)\ge\mathbb E_{q(z\mid x)}[\log p(x\mid z)]-\mathrm{KL}\big(q(z\mid x)\,\|\,p(z)\big).$$

**Recurrent network** — a feed-forward network with feedback loops, for sequences and time series (text, DNA,
video): $s_t=f(Ux_t+Ws_{t-1})$, $o_t=g(Vs_t)$. **Unrolling** over time gives a deep feed-forward network with the
same $U, W, V$ at every step, trained by backpropagation through time.

> 🎓 **Exam tip.** Explain why a bottleneck is necessary, what "linear autoencoder = PCA" means (same subspace, not necessarily
the same axes), the roles of encoder (inference) and decoder (generative model) in a VAE, and what "unrolling" an RNN means
(shared weights at every time step).

---
## 11 · Neural networks vs probabilistic graphical models, and what to read next

> 📍 **Where we are.** Part D of the course built *interpretable* models of how variables depend on each other
> (Bayesian networks, Gaussian graphical models, HMMs). How do they compare with neural networks?

| | Neural networks | Probabilistic graphical models |
|---|---|---|
| Strengths | very expressive; excellent at extracting features from raw data (sequence, images) | easy to interpret; model relationships among variables explicitly; handle missing data and uncertainty |
| Weaknesses | hard to interpret; need many samples; non-convex training | highly parametric: rely on explicit (often restrictive) assumptions; features must be chosen by hand |
| Example in this course | 1-D CNN for binding sites (Section 8) | signalling network from (simulated) Sachs-style data ([Module 17](17_bayesian_networks_inference_learning.ipynb)) |

The natural next step is to **combine the strengths of both**: a conditional random field (a graphical model over pixel
labels) on top of features learned by a CNN for image segmentation; a VAE whose latent code follows an HMM (for
sequences or behaviour) or a mixture model (for clustering cells). The three recurring ideas of the course reappear
in neural form: the **two-step template** (a network defines $P(Y\mid X;w)$; cross-entropy = MLE, weight decay = MAP),
**structure buys tractability** (weight tying in CNNs, GNNs and RNNs plays the role that conditional independence plays
in graphical models: it slashes the number of free parameters), and **latent variables** (autoencoders and VAEs).

For further reading the 02-620 lecture recommends, and we second, Zou *et al.* (2019), *A primer on deep learning in
genomics*, Nature Genetics — a short, readable tour of CNNs, RNNs, autoencoders and model interpretation for genomic data.

✏️ **Your turn.** Choose a biological data set you care about (single-cell RNA-seq, protein sequences, a time series of
physiological measurements, images of tissue sections …). Which architecture from this module would you try first, which
graphical model from Part D, and what does each one *assume* about the data? Which would you trust more with 50 samples,
and which with 50,000?

## 📌 Key takeaways

- **A neuron is logistic regression**: a weighted sum plus a sigmoid; its decision boundary is a line (hyperplane).
- **Hidden layers learn intermediate representations**, which makes nonlinear boundaries (XOR, epistasis) possible;
  sigmoid, tanh and ReLU activations differ mainly in saturation, and ReLU avoids vanishing gradients.
- **Learning = minimising a loss** (cross-entropy = negative log-likelihood; or the classic squared error) with
  **backpropagation**: forward pass, then error terms $\delta_k$, $\delta_h$ sent backwards through the same weights.
- **SGD and mini-batch SGD** make many cheap, noisy updates per epoch; mini-batches are the practical default.
- Networks have many parameters and non-convex losses: **overfitting** is controlled by network size, **early stopping**
  on a validation set and **weight decay**; different random starts can end in different minima.
- **Architectures encode assumptions**: CNNs tie weights across positions (a filter is a learnable PWM), GNNs across
  nodes, RNNs across time; autoencoders compress through a bottleneck (linear AE = PCA), VAEs make it probabilistic.
- Neural networks are expressive but data-hungry and hard to interpret; graphical models are interpretable but
  rigid — **hybrids** combine the two. In one sentence: various architectures, all learned by backpropagation and
  stochastic gradient descent.

---
## ✅ Checkpoint

1. Write a single sigmoid neuron with inputs $X_1,X_2,X_3$ as a probability model. Which course model is it, and how many parameters does it have?
   <details><summary>Answer</summary>$P(Y=1\mid X)=\sigma(w_0+w_1X_1+w_2X_2+w_3X_3)$ — logistic regression, with 4 parameters ($w_0,\dots,w_3$).</details>
2. Using the hand-built network of Section 2 (hidden intercepts −5 and −15, all input weights 10, output weights 10 and −20, output intercept −5), compute the output for genotype (1, 1).
   <details><summary>Answer</summary>Hidden: $\sigma(-5+20)=\sigma(15)\approx1$, $\sigma(-15+20)=\sigma(5)\approx0.993$. Output: $\sigma(-5+10\cdot1-20\cdot0.993)=\sigma(-14.9)\approx3\times10^{-7}$ — 'folds', as XOR requires.</details>
3. How many parameters does a fully connected network with 20 inputs, 10 hidden units and 3 outputs have?
   <details><summary>Answer</summary>$20\cdot10+10$ (hidden weights + intercepts) $+\,10\cdot3+3$ (output) $=243$.</details>
4. An output unit has $o_k=0.8$ and target $t_k=1$. Compute $\delta_k$ for the squared-error loss and for the cross-entropy loss. Which gives the larger update?
   <details><summary>Answer</summary>Squared error: $0.8\cdot0.2\cdot0.2=0.032$. Cross-entropy: $1-0.8=0.2$ — about six times larger, because the sigmoid slope $o_k(1-o_k)$ cancels.</details>
5. Why do very deep networks with sigmoid units train slowly, and how does ReLU help?
   <details><summary>Answer</summary>Backpropagation multiplies one activation slope per layer; the sigmoid slope is at most 0.25 and near 0 when saturated, so the product vanishes with depth. ReLU has slope exactly 1 for positive inputs, so gradients pass unchanged along active paths.</details>
6. With $N=1{,}000$ training samples, how many weight updates per epoch do full-batch GD, SGD and mini-batch SGD with $B=32$ make?
   <details><summary>Answer</summary>1, 1,000 and $\lceil1000/32\rceil=32$.</details>
7. Code reading: how many parameters has `nn.Conv1d(4, 8, kernel_size=8)`, and does the number change for 60-bp or 1,000-bp sequences? Compare with `nn.Linear(4 * 60, 32)`.
   <details><summary>Answer</summary>$8\cdot4\cdot8+8=264$, independent of sequence length (tied weights). `nn.Linear(240, 32)` has $240\cdot32+32=7{,}712$, and it grows with the sequence length.</details>
8. Why does a linear autoencoder with a 2-unit bottleneck reach the same reconstruction error as 2-component PCA, and why are its codes not identical to the PC scores?
   <details><summary>Answer</summary>With linear maps and squared error, the best rank-2 reconstruction is the projection onto the top-2 principal subspace (Eckart–Young / Baldi & Hornik), so the optimum is the same. Any invertible 2 × 2 transformation of the code that the decoder undoes gives the same reconstruction, so the axes are only determined up to such a transformation.</details>

## 🧠 Questions & discussion

1. A colleague trains a 3-hidden-layer network on expression profiles of 150 tumours (20,000 genes) and reports 98 % accuracy. What would you ask before believing it? (Think of Sections 6 and 10 and of Module 23.)
2. The learned CNN filter in Section 8 resembled the E-box. Give two reasons why, on real ChIP-seq data, a filter might look like a motif and still not mean that this transcription factor binds there.
3. Weight tying (CNN), conditional independence (naïve Bayes, Bayesian networks) and sparsity (lasso) all reduce the number of free parameters. In what sense are they the same idea, and how do they differ?
4. When would you prefer an HMM ([Module 19](19_hmm_inference.ipynb)) over an RNN for a DNA sequence problem, and vice versa?
5. Minimising cross-entropy is maximum likelihood. What prior corresponds to weight decay, and what would a prior that favours *sparse* weights look like?

## 🚀 Bonus — going further

- Make the CNN of Section 8 strand-aware: scan both the sequence and its reverse complement with the same filters and take the maximum (as the binding-site CNNs described at the start do), then plant a non-palindromic motif on random strands (e.g. the GATA site `AGATAA`, whose reverse complement is `TTATCT`) and compare with the one-strand model.
- Compute a **saliency map** for a test sequence: `x.requires_grad_(True)`, back-propagate the output logit, and plot `x.grad * x` along the sequence. Does it highlight the planted motif?
- Train a small VAE (encoder outputs $\mu$ and $\log\sigma^2$; use the reparameterisation $z=\mu+\sigma\varepsilon$) on 16 × 16 images of cell nuclei from the toolkit's `simulate_cell_images()` (copy it from `course_toolkit.py`) and generate new nuclei by decoding $z\sim\mathcal N(0,I)$; Level 2 [Module 27](27_variational_inference_vae.ipynb) develops VAEs in full.
- The sister course's PyTorch module, [BiologyToResearch Module 05](../../../../cryoem/tutorial/BiologyToResearch/notebooks/05_deep_learning_pytorch.ipynb), trains 2-D and 3-D CNNs on simulated cryo-electron-tomography particles, with data loaders, augmentation and a data-leakage experiment — the natural next step after this notebook.
- **Level 2 continues the architectures of this module:** attention and transformers ([Module 24](24_attention_transformers.ipynb)), diffusion and flow-matching generative models ([Module 28](28_diffusion_flow_matching.ipynb)), graph neural networks ([Module 25](25_graph_neural_networks.ipynb)) and deep learning on regulatory DNA ([Module 36](36_regulatory_genomics_deep_learning.ipynb)).
- Logistic regression as a generative-vs-discriminative baseline on breast-cancer data: [GenerativeDiscriminativeClassification](../../GenerativeDiscriminativeClassification/); and imaging AI more broadly: [sci_img tutorials](../../../../sci_img/tutorial/README.md).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 22 · Neural networks
Input: SIMULATED epistasis signals (2 loci), SIMULATED 60-bp sequences with/without an E-box, a toy
       drug–protein graph, SIMULATED leukaemia expression (72 patients, 1,000 most variable genes; shaped like
       Golub et al. 1999)
Output: decision boundaries, learning curves, gradient checks, a trained 1-D CNN and its filter,
        autoencoder codes vs PCA scores
Method: two-layer sigmoid network trained by backpropagation (NumPy, then PyTorch autograd);
        GD vs SGD vs mini-batch; 1-D CNN with global max pooling; GCN layer; linear/nonlinear autoencoders
Key equation: δ_k = o_k(1 − o_k)(t_k − o_k);  δ_h = o_h(1 − o_h) Σ_k w_hk δ_k;  w_ij ← w_ij + η δ_j x_i
Key parameters: hidden units, learning rate η, batch size, epochs, weight decay λ, filter width k
What changed when I modified … (hidden units / batch size / FIDELITY / filter width): …
One failure mode: (e.g. 50 hidden units on 40 samples overfit; nonlinear AE memorised 38 patients)
Connection to the 02-620 topics (which idea of the lecture this illustrates):
       neural networks (neuron = logistic regression, hidden layers, activations, CNN, GNN, AE/VAE, RNN,
       loss minimisation, SGD, backpropagation)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its lecture on neural networks.
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — Chapter 4 (artificial neural networks; the BACKPROPAGATION algorithm of Table 4.2 and the vowel-recognition example).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Chapter 11 (neural networks).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 16.5 (feed-forward neural networks) and Chapter 28 (deep learning).
- Goodfellow I., Bengio Y., Courville A. (2016) *Deep Learning*. MIT Press.
- Rumelhart D. E., Hinton G. E., Williams R. J. (1986) Learning representations by back-propagating errors. *Nature* 323:533–536.
- LeCun Y., Bottou L., Bengio Y., Haffner P. (1998) Gradient-based learning applied to document recognition. *Proc IEEE* 86:2278–2324 (LeNet-5).
- Alipanahi B., Delong A., Weirauch M. T., Frey B. J. (2015) Predicting the sequence specificities of DNA- and RNA-binding proteins by deep learning. *Nat Biotechnol* 33:831–838.
- Zhou J., Troyanskaya O. G. (2015) Predicting effects of noncoding variants with deep learning-based sequence model. *Nat Methods* 12:931–934.
- Kipf T. N., Welling M. (2017) Semi-supervised classification with graph convolutional networks. *ICLR*.
- Zitnik M., Agrawal M., Leskovec J. (2018) Modeling polypharmacy side effects with graph convolutional networks. *Bioinformatics* 34:i457–i466 (Decagon).
- Baldi P., Hornik K. (1989) Neural networks and principal component analysis: learning from examples without local minima. *Neural Networks* 2:53–58.
- Hinton G. E., Salakhutdinov R. R. (2006) Reducing the dimensionality of data with neural networks. *Science* 313:504–507 (deep autoencoders).
- Kingma D. P., Welling M. (2014) Auto-encoding variational Bayes. *ICLR*.
- Tipping M. E., Bishop C. M. (1999) Probabilistic principal component analysis. *J R Stat Soc B* 61:611–622.
- Lopez R., Regier J., Cole M. B., Jordan M. I., Yosef N. (2018) Deep generative modeling for single-cell transcriptomics. *Nat Methods* 15:1053–1058.
- Durbin R., Eddy S., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press (position weight matrices and log-odds scores).
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537 (the study the simulated leukaemia data imitate).
- Zou J., Huss M., Abid A., Mohammadi P., Torkamani A., Telenti A. (2019) A primer on deep learning in genomics. *Nat Genet* 51:12–18 (the further reading recommended by the 02-620 lecture).

## ➡️ Next

Every model in this course has knobs that are not learned by gradient descent — the number of hidden
units, λ, k in k-NN, the tree depth, the kernel of an SVM. How do we choose them without fooling ourselves?
**[Module 23 · Model selection: bias, variance, cross-validation and the limits of prediction](23_model_selection.ipynb)**
answers this, and finishes with the leukaemia classifier done honestly.